# Distribution-Based Semantic Watermarking Model

## Overview

This notebook implements a **distribution-based approach** to semantic watermarking, shifting from binary classification to probabilistic distance-based scoring. Instead of predicting discrete labels, the model learns class-conditional distributions in latent space and computes normalized distance scores for semantic drift detection.

## Key Concepts

### Traditional Classifier vs Distribution-Based Model

| Aspect | Traditional Classifier | Distribution Model (Ours) |
|--------|----------------------|---------------------------|
| **Output** | Binary labels (0/1) | Distance scores (continuous) |
| **Decision** | Hard boundary | Soft, probabilistic |
| **Interpretability** | Low (just label) | High (distances to all classes) |
| **Semantic Drift** | Cannot quantify | Quantifiable via distance changes |
| **Uncertainty** | Limited (softmax probs) | Rich (multi-class distances) |

### Architecture

```
CLIP Embedding (512-dim)
    ↓
Encoder (512 → 256 → 128)
    ↓
Latent Space (μ, log σ²)  ← Class Prototypes (μ_c, Σ_c)
    ↓
z = μ + σ ⊙ ε
    ↓
Distance Computation: d(z, c) = ||z - μ_c||_Σ
    ↓
Scores: softmax(-d / τ)
    ↓
Decoder (128 → 256 → 512)
```

### Multi-Component Loss

**L_total = λ_recon · L_recon + λ_class · L_class + λ_dist · L_dist + λ_KL · L_KL**

- **L_recon**: Cosine similarity loss (reconstruction quality)
- **L_class**: Cross-entropy on distance-based scores
- **L_dist**: Contrastive distance loss (intra-class compactness)
- **L_KL**: Latent space regularization

## Dataset

- **Classes**: Normal, Violence, Sexual (3 classes)
- **Features**: CLIP ViT-B/32 embeddings (512-dim)
- **Task**: Detect semantic drift in manipulated images

## Goals

1. Learn distinct class distributions in latent space
2. Provide interpretable distance-based semantic scores
3. Quantify semantic drift for watermarking
4. Compare with classifier baseline (~89% F1)


In [ ]:
import google.colab
google.colab.drive.mount('/content/drive')

In [ ]:
# semantic_wm 데이터셋 압축 해제
zip_path = '/content/drive/MyDrive/semantic_wm/dataset.zip'

print("압축 해제 중...")
!unzip -q {zip_path} -d /content/semantic_wm/
print("✅ 데이터셋 압축 해제 완료")

# 데이터셋 구조 확인
print("\n=== 데이터셋 구조 ===")
!ls -R /content/semantic_wm/dataset/train/

In [ ]:
# ============================================================================
# IMPORTS AND SETUP
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from torch.cuda.amp import autocast
from transformers import CLIPProcessor, CLIPModel  # CLIP 모델 추가

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from scipy.spatial.distance import mahalanobis
from scipy.stats import chi2, pearsonr

import pandas as pd
from tqdm import tqdm
import copy
import os
import json
import pickle
from datetime import datetime
from pathlib import Path
from PIL import Image
from collections import defaultdict

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Color scheme for consistent visualization
CLASS_COLORS = {
    0: '#2ecc71',  # Normal - Green
    1: '#e74c3c',  # Violence - Red
    2: '#9b59b6'   # Sexual - Purple
}
CLASS_NAMES = ['Normal', 'Violence', 'Sexual']

# Create output directory for plots
os.makedirs('plots', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('embeddings', exist_ok=True)
print("✅ Setup complete")

In [ ]:
# ============================================================================
# HYPERPARAMETERS
# ============================================================================

# Model Architecture
INPUT_DIM = 512        # CLIP embedding dimension
LATENT_DIM = 64        # Latent space dimension
HIDDEN_DIM = 256       # Hidden layer dimension
NUM_CLASSES = 3        # Normal, Violence, Sexual

# Training Configuration
BATCH_SIZE = 64
LEARNING_RATE = 5e-4
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15

# Loss Weights (4 components - no reconstruction)
LAMBDA_CLASS = 1.0     # Classification loss
LAMBDA_KL = 0.01       # KL divergence
LAMBDA_SCL = 0.5       # Supervised Contrastive Loss


# SCL + Prototype EMA Configuration
USE_SCL = True         # Enable Supervised Contrastive Loss
USE_PROTO_EMA = True   # Enable Prototype EMA updates
SCL_TEMPERATURE = 0.07 # Temperature for SCL
EMA_MOMENTUM = 0.9     # Momentum for Prototype EMA

# Distance-to-Score Conversion
TEMPERATURE = 1.0      # Temperature for softmax over distances

# Training/Validation Split
TRAIN_VAL_SPLIT = 0.8

print("Hyperparameters:")
print(f"  Model: PrototypeEncoder (no decoder)")
print(f"  Input Dim: {INPUT_DIM}")
print(f"  Latent Dim: {LATENT_DIM}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Loss Weights: Class={LAMBDA_CLASS}, KL={LAMBDA_KL}, SCL={LAMBDA_SCL}")
print(f"  SCL Enabled: {USE_SCL} (temp={SCL_TEMPERATURE})")
print(f"  Prototype EMA Enabled: {USE_PROTO_EMA} (momentum={EMA_MOMENTUM})")
print("\n⚠️  Reconstruction loss removed (decoder not used)")

## Dataset Configuration

Configure dataset paths. Update these paths to point to your image dataset.

Dataset structure:
```
dataset/
├── train/
│   ├── normal/
│   ├── sexual/
│   └── violence/
└── test/
    ├── normal/
    ├── sexual/
    └── violence/
```

In [ ]:
# ============================================================================
# DATASET PATHS CONFIGURATION
# ============================================================================

# 📁 UPDATE THESE PATHS TO YOUR DATASET LOCATION
# For Google Colab (semantic_wm dataset, unzipped to /content/semantic_wm/):
#   train_dir = "/content/semantic_wm/dataset/train"
#   test_dir = "/content/semantic_wm/dataset/test"
# For local:
#   train_dir = "./dataset/train"
#   test_dir = "./dataset/test"

train_dir = "/content/semantic_wm/dataset/train"
test_dir = "/content/semantic_wm/dataset/test"

print(f"Dataset directories:")
print(f"  Train: {train_dir}")
print(f"  Test: {test_dir}")

In [ ]:
# ============================================================================
# IMAGE DATASET CLASS
# ============================================================================

class ImageDataset(Dataset):
    """
    Custom dataset for loading images from folders

    Structure:
        root/
        ├── normal/
        ├── sexual/
        └── violence/
    """

    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform

        # Class mapping
        self.class_to_idx = {'normal': 0, 'sexual': 1, 'violence': 2}
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

        # Load image paths
        self.samples = []
        self.class_counts = defaultdict(int)

        for class_name in ['normal', 'sexual', 'violence']:
            class_dir = self.root_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*'):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.gif', '.bmp']:
                        self.samples.append((img_path, self.class_to_idx[class_name]))
                        self.class_counts[class_name] += 1

        print(f"Loaded {len(self.samples)} images from {root_dir}")
        for class_name, count in self.class_counts.items():
            print(f"  {class_name}: {count}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # Return a black image as fallback
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        return image, label


print("✅ ImageDataset class defined")

In [ ]:
# ============================================================================
# LOAD DATASETS
# ============================================================================

print("Loading image datasets...")
train_dataset = ImageDataset(train_dir)
test_dataset = ImageDataset(test_dir)

print(f"\nTotal samples:")
print(f"  Train: {len(train_dataset)}")
print(f"  Test: {len(test_dataset)}")

In [ ]:
# ============================================================================
# CLIP MODEL & EMBEDDING EXTRACTION (GPU Optimized)
# ============================================================================

# Load CLIP model with optimizations
print("Loading CLIP model...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Set to evaluation mode
clip_model.eval()

# GPU Optimizations
if device.type == 'cuda':
    # FP16 for faster inference and lower memory
    clip_model = clip_model.half()

    # Enable TF32 for Ampere GPUs (RTX 30xx, A100, etc.)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    # Enable cudnn benchmark for consistent input sizes
    torch.backends.cudnn.benchmark = True

    # Try torch.compile for PyTorch 2.0+ (significant speedup)
    try:
        if hasattr(torch, 'compile'):
            clip_model = torch.compile(clip_model, mode='reduce-overhead')
            print("✅ torch.compile enabled")
    except Exception as e:
        print(f"⚠️ torch.compile not available: {e}")

print("✅ CLIP model loaded with GPU optimization")


def custom_collate_fn(batch):
    """Custom collate function to handle PIL Images"""
    images = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])
    return images, labels


def extract_clip_embeddings_optimized(dataset, batch_size=64):
    """
    Extract CLIP embeddings with GPU optimization

    Optimizations:
    1. Prefetching with pin_memory and non_blocking transfers
    2. CUDA streams for async operations
    3. Efficient memory management
    4. Updated autocast API (no deprecation warning)
    5. Batch processing optimization
    """

    # Optimized DataLoader settings
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2 if device.type == 'cuda' else 0,  # Enable workers for GPU
        pin_memory=True if device.type == 'cuda' else False,
        prefetch_factor=2 if device.type == 'cuda' else None,  # Prefetch batches
        persistent_workers=True if device.type == 'cuda' else False,  # Reuse workers
        collate_fn=custom_collate_fn
    )

    embeddings = []
    labels = []

    # Create CUDA stream for async operations
    if device.type == 'cuda':
        stream = torch.cuda.Stream()

    with torch.no_grad():
        for batch_idx, (images, lbls) in enumerate(tqdm(dataloader, desc="Extracting CLIP embeddings")):
            if len(images) == 0:
                continue

            try:
                # Process images with CLIP processor
                inputs = clip_processor(
                    images=images,
                    return_tensors="pt",
                    padding=True
                )
            except Exception as e:
                print(f"Error processing batch {batch_idx}: {e}")
                continue

            if device.type == 'cuda':
                # Use CUDA stream for async transfer
                with torch.cuda.stream(stream):
                    # Non-blocking transfer to GPU
                    inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

                    # Wait for transfer to complete
                    stream.synchronize()

                    # Extract embeddings with mixed precision (updated API)
                    with torch.amp.autocast('cuda'):
                        emb = clip_model.get_image_features(**inputs)

                    # Normalize embeddings (still in FP16)
                    emb = emb / emb.norm(dim=-1, keepdim=True)

                    # Convert to FP32 and move to CPU asynchronously
                    emb = emb.float().cpu()
            else:
                # CPU path
                inputs = {k: v.to(device) for k, v in inputs.items()}
                emb = clip_model.get_image_features(**inputs)
                emb = emb / emb.norm(dim=-1, keepdim=True)
                emb = emb.cpu()

            embeddings.append(emb)
            labels.append(lbls)

            # Efficient memory management - less frequent but still effective
            if batch_idx % 20 == 0 and device.type == 'cuda':
                torch.cuda.empty_cache()

    if len(embeddings) == 0:
        raise ValueError("No embeddings extracted! Dataset may be empty.")

    # Final concatenation
    embeddings = torch.cat(embeddings, dim=0)
    labels = torch.cat(labels, dim=0)

    return embeddings, labels


# Legacy function for compatibility
def extract_clip_embeddings(dataset, batch_size=64):
    """Wrapper for backward compatibility"""
    return extract_clip_embeddings_optimized(dataset, batch_size)


# Determine optimal batch size based on GPU memory
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory
    gpu_name = torch.cuda.get_device_name(0)

    # Larger batch sizes for better GPU utilization
    if gpu_memory > 16e9:  # > 16GB (A100, etc.)
        clip_batch_size = 256
    elif gpu_memory > 8e9:  # > 8GB (RTX 3070+, etc.)
        clip_batch_size = 128
    elif gpu_memory > 4e9:  # > 4GB
        clip_batch_size = 64
    else:
        clip_batch_size = 32

    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_memory / 1e9:.1f} GB")
else:
    clip_batch_size = 32

print(f"Using batch size: {clip_batch_size}")

# Extract embeddings with progress tracking
import time

print(f"\nExtracting train embeddings...")
start_time = time.time()
X_train, y_train = extract_clip_embeddings_optimized(train_dataset, batch_size=clip_batch_size)
train_time = time.time() - start_time
print(f"  Time: {train_time:.1f}s ({len(X_train)/train_time:.1f} samples/sec)")

print(f"\nExtracting test embeddings...")
start_time = time.time()
X_test, y_test = extract_clip_embeddings_optimized(test_dataset, batch_size=clip_batch_size)
test_time = time.time() - start_time
print(f"  Time: {test_time:.1f}s ({len(X_test)/test_time:.1f} samples/sec)")

print(f"\n✅ CLIP embeddings extracted")
print(f"  Train: {X_train.shape}, Labels: {y_train.shape}")
print(f"  Test: {X_test.shape}, Labels: {y_test.shape}")

# Cleanup
if device.type == 'cuda':
    torch.cuda.empty_cache()
    print(f"  GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f}GB / {torch.cuda.max_memory_allocated()/1e9:.2f}GB")

# Save embeddings for future use
print(f"\nSaving embeddings to embeddings/ directory...")
torch.save(X_train, 'embeddings/X_train.pt')
torch.save(y_train, 'embeddings/y_train.pt')
torch.save(X_test, 'embeddings/X_test.pt')
torch.save(y_test, 'embeddings/y_test.pt')
print("✅ Embeddings saved")


In [ ]:
# ============================================================================
# LABEL REMAPPING & DATA SPLITTING
# ============================================================================

# Verify data format
assert X_train.shape[1] == INPUT_DIM, f"Expected {INPUT_DIM}-dim embeddings, got {X_train.shape[1]}"
assert y_train.dim() == 1, f"Expected 1D labels, got {y_train.dim()}D"

# Check class labels
unique_labels = torch.unique(y_train).tolist()
print(f"Original unique labels: {unique_labels}")
assert set(unique_labels) == {0, 1, 2}, f"Expected labels {{0, 1, 2}}, got {unique_labels}"

# IMPORTANT: Label remapping for consistency
# ImageDataset uses: normal=0, sexual=1, violence=2
# Distribution model expects: Normal=0, Violence=1, Sexual=2
# Therefore: swap sexual (1) ↔ violence (2)
print("\n⚠️  Remapping class labels for consistency:")
print("  Original: normal=0, sexual=1, violence=2")
print("  Remapped: Normal=0, Violence=1, Sexual=2")

label_mapping = {0: 0, 1: 2, 2: 1}  # Swap sexual and violence
y_train = torch.tensor([label_mapping[l.item()] for l in y_train])
y_test = torch.tensor([label_mapping[l.item()] for l in y_test])
print("✅ Labels remapped successfully")

# Print class distribution
print("\nClass distribution (after remapping):")
for i, class_name in enumerate(CLASS_NAMES):
    train_count = (y_train == i).sum().item()
    test_count = (y_test == i).sum().item()
    print(f"  {class_name}:")
    print(f"    Train: {train_count} ({100*train_count/len(y_train):.1f}%)")
    print(f"    Test: {test_count} ({100*test_count/len(y_test):.1f}%)")

# Create stratified train/val split
print(f"\nCreating stratified train/val split ({int(TRAIN_VAL_SPLIT*100)}/{int((1-TRAIN_VAL_SPLIT)*100)})...")

X_train_np = X_train.numpy()
y_train_np = y_train.numpy()

X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_np, y_train_np,
    test_size=1-TRAIN_VAL_SPLIT,
    stratify=y_train_np,
    random_state=SEED
)

# Convert back to tensors
X_train_split = torch.from_numpy(X_train_split).float()
X_val = torch.from_numpy(X_val).float()
y_train_split = torch.from_numpy(y_train_split).long()
y_val = torch.from_numpy(y_val).long()

print(f"\nDataset split:")
print(f"  Train: {len(X_train_split)} samples")
print(f"  Val: {len(X_val)} samples")
print(f"  Test: {len(X_test)} samples")

# Verify stratification
print("\nClass distribution in splits:")
for split_name, split_labels in [("Train", y_train_split), ("Val", y_val), ("Test", y_test)]:
    print(f"  {split_name}:")
    for i, class_name in enumerate(CLASS_NAMES):
        count = (split_labels == i).sum().item()
        pct = 100 * count / len(split_labels)
        print(f"    {class_name}: {count} ({pct:.1f}%)")

# Create DataLoaders
print(f"\nCreating DataLoaders (batch_size={BATCH_SIZE})...")

train_dataset = TensorDataset(X_train_split, y_train_split)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)

print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

print("\n" + "="*70)
print("✅ DATA LOADING COMPLETE")
print("="*70)

## Model Architecture

### Variational Autoencoder with Class-Conditional Distributions

The model consists of:
1. **Encoder**: Maps CLIP embeddings to latent distributions
2. **Class Prototypes**: Learnable mean and covariance for each class
3. **Distance Computer**: Computes Mahalanobis distances
4. **Decoder**: Reconstructs CLIP embeddings


In [ ]:
# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

class PrototypeEncoder(nn.Module):
    """
    Prototype-based Encoder for Semantic Watermarking

    This model learns to:
    1. Encode CLIP embeddings into a latent space
    2. Maintain learnable prototype distributions for each class
    3. Compute distances from latent points to class prototypes

    Note: No decoder - reconstruction is not used in this approach.
    This reduces parameters by ~50% and improves CDAS.
    """

    def __init__(self, input_dim=512, latent_dim=64, hidden_dim=256, num_classes=3):
        super(PrototypeEncoder, self).__init__()

        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes

        # ====================================================================
        # Encoder: CLIP embedding → Latent distribution parameters
        # ====================================================================
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU()
        )

        # Latent distribution parameters
        self.fc_mu = nn.Linear(hidden_dim // 4, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 4, latent_dim)

        # ====================================================================
        # Class Prototypes: Learnable mean and covariance for each class
        # ====================================================================
        # Prototype means: (num_classes, latent_dim)
        self.prototype_means = nn.Parameter(
            torch.randn(num_classes, latent_dim) * 0.1
        )

        # Prototype log-variances: (num_classes, latent_dim)
        # We use diagonal covariance matrices for efficiency
        self.prototype_logvars = nn.Parameter(
            torch.zeros(num_classes, latent_dim)
        )

        # Temperature for distance-to-score conversion
        self.temperature = nn.Parameter(torch.tensor(TEMPERATURE))

    def encode(self, x):
        """Encode input to latent distribution parameters"""
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = μ + σ ⊙ ε"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def compute_mahalanobis_distances(self, z):
        """
        Compute Mahalanobis distance from z to each class prototype

        For diagonal covariance: d² = Σ((z - μ)² / σ²)

        Args:
            z: (batch_size, latent_dim)

        Returns:
            distances: (batch_size, num_classes)
        """
        batch_size = z.shape[0]
        distances = []

        for c in range(self.num_classes):
            # Get prototype parameters
            mu_c = self.prototype_means[c]  # (latent_dim,)
            var_c = torch.exp(self.prototype_logvars[c])  # (latent_dim,)

            # Compute squared Mahalanobis distance
            diff = z - mu_c.unsqueeze(0)  # (batch_size, latent_dim)
            maha_sq = ((diff ** 2) / (var_c.unsqueeze(0) + 1e-6)).sum(dim=1)  # (batch_size,)

            distances.append(torch.sqrt(maha_sq + 1e-6))

        distances = torch.stack(distances, dim=1)  # (batch_size, num_classes)
        return distances

    def distances_to_scores(self, distances):
        """
        Convert distances to probability-like scores using softmax

        score_c = exp(-d_c / τ) / Σ exp(-d_c' / τ)

        Lower distance → Higher score
        """
        neg_distances = -distances / self.temperature
        scores = F.softmax(neg_distances, dim=1)
        return scores

    def forward(self, x, return_all=False):
        """
        Forward pass

        Args:
            x: Input CLIP embeddings (batch_size, input_dim)
            return_all: If True, return all intermediate values

        Returns:
            mu, logvar, z, distances, scores
        """
        # Encode
        mu, logvar = self.encode(x)

        # Sample latent vector
        z = self.reparameterize(mu, logvar)

        # Compute distances to prototypes
        distances = self.compute_mahalanobis_distances(z)

        # Convert to scores
        scores = self.distances_to_scores(distances)

        if return_all:
            return {
                'mu': mu,
                'logvar': logvar,
                'z': z,
                'distances': distances,
                'scores': scores,
                'prototype_means': self.prototype_means,
                'prototype_vars': torch.exp(self.prototype_logvars)
            }
        else:
            return mu, logvar, z, distances, scores


# Alias for backward compatibility
DistributionVAE = PrototypeEncoder


# Initialize model
model = PrototypeEncoder(
    input_dim=INPUT_DIM,
    latent_dim=LATENT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Architecture: PrototypeEncoder (no decoder)")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"\nPrototype shapes:")
print(f"  Means: {model.prototype_means.shape}")
print(f"  Log-vars: {model.prototype_logvars.shape}")
print("\n✅ Model initialized (decoder removed, ~50% parameter reduction)")

## Loss Functions

### Multi-Component Loss

**L_total = λ_class · L_class + λ_SCL · L_SCL**



-**Classification Loss (L_class)**: Cross-entropy on scores
   - Trains the model to assign high scores to correct class
   - Uses distance-based scores


In [ ]:
# ============================================================================
# LOSS FUNCTIONS (No Reconstruction)
# ============================================================================
#
# Components:
# - Classification loss: Cross-entropy with label smoothing
# - KL divergence loss: Regularize latent space
# - Supervised Contrastive Loss (SCL): Class-wise separation in latent space
#
# Note: Reconstruction loss removed - not needed for prototype-based approach
# ============================================================================

def classification_loss(scores, labels, label_smoothing=0.1):
    """
    Cross-entropy loss on distance-based scores with label smoothing

    Label smoothing helps with calibration by preventing overconfident predictions.

    Args:
        scores: (batch_size, num_classes) - probability-like scores
        labels: (batch_size,) - true class labels
        label_smoothing: smoothing factor (default: 0.1)
    """
    return F.cross_entropy(scores, labels, label_smoothing=label_smoothing)




def kl_divergence_loss(mu, logvar):
    """
    KL divergence: KL(q(z|x) || p(z))

    Regularizes the latent space toward standard normal.
    Even without decoder, this helps with:
    - Smooth latent space
    - Prevents posterior collapse
    - Works well with SCL for structured latent space
    """
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return kl


# ============================================================================
# Supervised Contrastive Loss (SCL)
# ============================================================================

class SupervisedContrastiveLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., 2020)

    Key idea: Pull same-class samples together, push different-class samples apart
    in the latent space.

    L = -Σ_i (1/|P(i)|) Σ_{p∈P(i)} log[exp(z_i·z_p/τ) / Σ_{a∈A(i)} exp(z_i·z_a/τ)]

    Where:
    - P(i): positive samples (same class as i, excluding self)
    - A(i): all samples except i
    - τ: temperature parameter (default: 0.07)

    Benefits:
    - Better class separation in latent space
    - More robust representations
    - Improved distance-confidence alignment
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        """
        Args:
            features: (batch_size, feature_dim) - latent vectors z
            labels: (batch_size,) - class labels

        Returns:
            loss: scalar supervised contrastive loss
        """
        device = features.device
        batch_size = features.size(0)

        # L2 normalize features for cosine similarity
        features = F.normalize(features, dim=1)

        # Compute similarity matrix: (batch_size, batch_size)
        sim_matrix = torch.matmul(features, features.T) / self.temperature

        # Create mask for positive pairs (same class, excluding self)
        labels = labels.contiguous().view(-1, 1)
        mask_positive = torch.eq(labels, labels.T).float().to(device)
        mask_self = torch.eye(batch_size, device=device)
        mask_positive = mask_positive - mask_self  # Remove self from positives

        # For numerical stability, subtract max
        logits_max, _ = torch.max(sim_matrix, dim=1, keepdim=True)
        logits = sim_matrix - logits_max.detach()

        # Compute log_prob
        exp_logits = torch.exp(logits)

        # Mask out self-comparisons
        exp_logits = exp_logits * (1 - mask_self)

        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-6)

        # Compute mean of log-likelihood over positive pairs
        # For each anchor, average over its positive samples
        num_positives = mask_positive.sum(dim=1)

        # Avoid division by zero (samples with no positives in batch)
        num_positives = torch.clamp(num_positives, min=1)

        mean_log_prob_pos = (mask_positive * log_prob).sum(dim=1) / num_positives

        # Loss is negative of mean log prob
        loss = -mean_log_prob_pos.mean()

        return loss


# ============================================================================
# Prototype EMA Manager
# ============================================================================

class PrototypeEMAManager:
    """
    Exponential Moving Average update for class prototypes

    Instead of only learning prototypes through backprop,
    we also update them using EMA of actual class centers.

    prototype_new = momentum * prototype_old + (1 - momentum) * batch_center

    Benefits:
    - More stable prototype positions
    - Faster convergence
    - Better alignment with actual data distribution
    """
    def __init__(self, momentum=0.9):
        self.momentum = momentum
        self.initialized = False

    @torch.no_grad()
    def update(self, model, z, labels, num_classes):
        """
        Update prototype means using EMA

        Args:
            model: The PrototypeEncoder model
            z: Latent vectors from current batch (batch_size, latent_dim)
            labels: Class labels (batch_size,)
            num_classes: Number of classes
        """
        for c in range(num_classes):
            mask = (labels == c)
            if mask.sum() > 0:
                # Compute batch center for this class
                class_center = z[mask].mean(dim=0)

                # EMA update
                model.prototype_means.data[c] = (
                    self.momentum * model.prototype_means.data[c] +
                    (1 - self.momentum) * class_center
                )

    def reset(self):
        """Reset the manager state"""
        self.initialized = False


# Initialize SCL loss function
scl_loss_fn = SupervisedContrastiveLoss(temperature=SCL_TEMPERATURE)

# Initialize Prototype EMA manager
prototype_ema = PrototypeEMAManager(momentum=EMA_MOMENTUM)


# ============================================================================
# Total Loss Computation
# ============================================================================

def compute_total_loss(model, x, labels, lambda_class=1.0,
                       lambda_kl=0.01, lambda_scl=0.5,
                       use_scl=True, use_proto_ema=True):
    """
    Compute total loss with all components.

    Note: No reconstruction loss - decoder has been removed.

    Components:
    - Classification loss: cross-entropy with label smoothing (0.1)
    - Distance contrastive loss: simplified version
    - KL divergence loss
    - Supervised Contrastive Loss (SCL)
    - Prototype EMA updates

    Args:
        model: PrototypeEncoder model
        x: input batch
        labels: ground truth labels
        lambda_class: weight for classification loss
                lambda_kl: weight for KL divergence loss
        lambda_scl: weight for supervised contrastive loss
        use_scl: whether to use SCL (default: True)
        use_proto_ema: whether to update prototypes with EMA (default: True)

    Returns:
        total_loss: scalar
        loss_dict: dictionary with individual loss values
    """
    # Forward pass (no reconstruction)
    mu, logvar, z, distances, scores = model(x)

    # Compute individual losses
    l_class = classification_loss(scores, labels, label_smoothing=0.1)
    l_kl = kl_divergence_loss(mu, logvar)

    # Supervised Contrastive Loss
    if use_scl:
        l_scl = scl_loss_fn(z, labels)
    else:
        l_scl = torch.tensor(0.0, device=x.device)

    # Total loss (no reconstruction!)
    total_loss = (
        lambda_class * l_class +
        lambda_kl * l_kl +
        lambda_scl * l_scl
    )

    # Update prototypes with EMA (after loss computation, before backward)
    if use_proto_ema:
        prototype_ema.update(model, z.detach(), labels, NUM_CLASSES)

    # Loss dictionary for logging
    loss_dict = {
        'total': total_loss.item(),
        'class': l_class.item(),
        'kl': l_kl.item(),
        'scl': l_scl.item() if use_scl else 0.0
    }

    return total_loss, loss_dict


print("✅ Loss functions defined (reconstruction removed)")
print("\nComponents:")
print("  • Classification loss (cross-entropy + label smoothing)")
print("")
print("  • KL divergence loss")
print("  • Supervised Contrastive Loss (SCL)")
print("  • Prototype EMA updates")
print("\n⚠️  Reconstruction loss removed - not needed for prototype-based approach")

## Training Loop

Training with:
- Adam optimizer
- Learning rate scheduling (ReduceLROnPlateau)
- Early stopping based on validation F1
- Gradient clipping for stability
- Comprehensive logging


In [ ]:
# ============================================================================
# TRAINING LOOP (No Reconstruction)
# ============================================================================

def train_epoch(model, train_loader, optimizer, device,
                lambda_class, lambda_kl, lambda_scl,
                use_scl=True, use_proto_ema=True):
    """
    Train for one epoch with SCL and Prototype EMA.

    Args:
        model: PrototypeEncoder model
        train_loader: training data loader
        optimizer: optimizer
        device: torch device
        lambda_class, lambda_kl, lambda_scl: loss weights
        use_scl: whether to use Supervised Contrastive Loss
        use_proto_ema: whether to update prototypes with EMA
    """
    model.train()

    epoch_losses = {
        'total': [],
        'class': [],
        'kl': [],
        'scl': []
    }

    for batch_x, batch_y in tqdm(train_loader, desc="Training", leave=False):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Forward pass
        optimizer.zero_grad()
        total_loss, loss_dict = compute_total_loss(
            model, batch_x, batch_y,
            lambda_class=lambda_class,
            lambda_kl=lambda_kl,
            lambda_scl=lambda_scl,
            use_scl=use_scl,
            use_proto_ema=use_proto_ema
        )

        # Backward pass
        total_loss.backward()

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Log losses
        for key in epoch_losses:
            epoch_losses[key].append(loss_dict[key])

    # Average losses
    avg_losses = {key: np.mean(vals) for key, vals in epoch_losses.items()}
    return avg_losses


def evaluate(model, data_loader, device,
             lambda_class, lambda_kl, lambda_scl=0.5):
    """
    Evaluate model on validation/test set.

    Note: SCL and Prototype EMA are disabled during evaluation.
    """
    model.eval()

    epoch_losses = {
        'total': [],
        'class': [],
        'kl': [],
        'scl': []
    }

    all_labels = []
    all_predictions = []
    all_distances = []
    all_scores = []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            # Forward pass (disable EMA during eval)
            total_loss, loss_dict = compute_total_loss(
                model, batch_x, batch_y,
                lambda_class=lambda_class,
                    lambda_kl=lambda_kl,
                lambda_scl=lambda_scl,
                use_scl=True,
                use_proto_ema=False  # Don't update prototypes during eval
            )

            # Get predictions (minimum distance)
            mu, logvar, z, distances, scores = model(batch_x)
            predictions = distances.argmin(dim=1)

            # Store results
            for key in epoch_losses:
                epoch_losses[key].append(loss_dict[key])

            all_labels.append(batch_y.cpu())
            all_predictions.append(predictions.cpu())
            all_distances.append(distances.cpu())
            all_scores.append(scores.cpu())

    # Concatenate results
    all_labels = torch.cat(all_labels)
    all_predictions = torch.cat(all_predictions)
    all_distances = torch.cat(all_distances)
    all_scores = torch.cat(all_scores)

    # Compute metrics
    avg_losses = {key: np.mean(vals) for key, vals in epoch_losses.items()}

    accuracy = accuracy_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions, average='macro')
    precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_predictions, average='macro', zero_division=0)

    metrics = {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

    return avg_losses, metrics, all_labels, all_predictions, all_distances, all_scores


def train_model(model, train_loader, val_loader, num_epochs=100, learning_rate=5e-4,
                lambda_class=1.0, lambda_kl=0.01,
                lambda_scl=0.5, use_scl=True, use_proto_ema=True,
                early_stopping_patience=15):
    """
    Full training loop with SCL, Prototype EMA, early stopping and LR scheduling.

    Note: No reconstruction loss - using prototype-based approach only.

    Components:
    - Supervised Contrastive Loss (SCL) for better class separation
    - Prototype EMA for stable prototype updates
    - Label smoothing in classification loss
    - KL regularization for smooth latent space

    Args:
        model: PrototypeEncoder model
        train_loader: training data loader
        val_loader: validation data loader
        num_epochs: maximum number of epochs
        learning_rate: initial learning rate
        lambda_class, lambda_kl, lambda_scl: loss weights
        use_scl: whether to use Supervised Contrastive Loss (default: True)
        use_proto_ema: whether to use Prototype EMA (default: True)
        early_stopping_patience: patience for early stopping
    """
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    # Reset Prototype EMA state for fresh training
    if use_proto_ema:
        prototype_ema.reset()

    history = {
        'train_loss': [],
        'val_loss': [],
        'train_f1': [],
        'val_f1': [],
        'val_accuracy': [],
        'learning_rates': [],
        'scl_loss': []
    }

    best_val_f1 = 0.0
    best_model_state = None
    patience_counter = 0

    print("\n" + "="*70)
    print("TRAINING STARTED")
    print("="*70)
    print(f"  Model: PrototypeEncoder (no decoder)")
    print(f"  SCL enabled: {use_scl} (λ_scl = {lambda_scl})")
    print(f"  Prototype EMA enabled: {use_proto_ema}")
    print(f"  Label smoothing: 0.1")
    print(f"  Loss weights: class={lambda_class}, kl={lambda_kl}")
    print("="*70)

    for epoch in range(num_epochs):
        # Train
        train_losses = train_epoch(
            model, train_loader, optimizer, device,
            lambda_class, lambda_kl, lambda_scl,
            use_scl=use_scl, use_proto_ema=use_proto_ema
        )

        # Validate
        val_losses, val_metrics, _, _, _, _ = evaluate(
            model, val_loader, device,
            lambda_class, lambda_kl, lambda_scl
        )

        # Quick train metrics (on a subset for speed)
        with torch.no_grad():
            model.eval()
            train_subset_x, train_subset_y = next(iter(train_loader))
            train_subset_x = train_subset_x.to(device)
            train_subset_y = train_subset_y.to(device)
            mu, logvar, z, train_distances, _ = model(train_subset_x)
            train_preds = train_distances.argmin(dim=1).cpu()
            train_f1 = f1_score(train_subset_y.cpu(), train_preds, average='weighted')

        # Log history
        history['train_loss'].append(train_losses['total'])
        history['val_loss'].append(val_losses['total'])
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_metrics['f1'])
        history['val_accuracy'].append(val_metrics['accuracy'])
        history['learning_rates'].append(optimizer.param_groups[0]['lr'])
        history['scl_loss'].append(train_losses['scl'])

        # Learning rate scheduling
        scheduler.step(val_metrics['f1'])

        # Early stopping check
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}]")
            print(f"  Train Loss: {train_losses['total']:.4f} | Val Loss: {val_losses['total']:.4f}")
            print(f"  SCL Loss: {train_losses['scl']:.4f}")
            print(f"  Train F1: {train_f1:.4f} | Val F1: {val_metrics['f1']:.4f} | Val Acc: {val_metrics['accuracy']:.4f}")
            print(f"  LR: {optimizer.param_groups[0]['lr']:.6f} | Patience: {patience_counter}/{early_stopping_patience}")
            print("-" * 70)


        # t-SNE Visualization every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"\nGenerating t-SNE visualization for epoch {epoch+1}...")
                model.eval()
                all_z = []
                all_labels = []

                with torch.no_grad():
                    for batch_x, batch_y in val_loader:
                        batch_x = batch_x.to(device)
                        mu, logvar, z, distances, scores = model(batch_x)
                        all_z.append(z.cpu().numpy())
                        all_labels.append(batch_y.cpu().numpy())

                all_z = np.concatenate(all_z, axis=0)
                all_labels = np.concatenate(all_labels, axis=0)

                # Compute t-SNE
                from sklearn.manifold import TSNE
                tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_z)-1), n_iter=1000)
                z_tsne = tsne.fit_transform(all_z)

                # Calculate prototype positions (class centroids)
                prototype_tsne = []
                for c in range(NUM_CLASSES):
                    mask = all_labels == c
                    if mask.sum() > 0:
                        centroid = z_tsne[mask].mean(axis=0)
                        prototype_tsne.append(centroid)
                prototype_tsne = np.array(prototype_tsne)

                # Plot t-SNE visualization
                fig, ax = plt.subplots(figsize=(10, 8))

                # Plot data points colored by class
                for c in range(NUM_CLASSES):
                    mask = all_labels == c
                    if mask.sum() > 0:
                        ax.scatter(z_tsne[mask, 0], z_tsne[mask, 1],
                                  c=CLASS_COLORS[c], label=CLASS_NAMES[c],
                                  alpha=0.6, s=30, edgecolors='none')

                # Plot prototypes as large stars
                for c in range(NUM_CLASSES):
                    if c < len(prototype_tsne):
                        ax.scatter(prototype_tsne[c, 0], prototype_tsne[c, 1],
                                  c=CLASS_COLORS[c], marker='*', s=500,
                                  edgecolors='black', linewidths=2, zorder=10,
                                  label=f'{CLASS_NAMES[c]} Prototype')

                ax.set_title(f'Latent Space t-SNE - Epoch {epoch+1}', fontsize=14, fontweight='bold')
                ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
                ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
                ax.legend(loc='best', fontsize=9)
                ax.grid(True, alpha=0.3)
                plt.tight_layout()

                # Save visualization
                os.makedirs('plots', exist_ok=True)
                save_path = f'plots/tsne_epoch_{epoch+1:03d}.png'
                plt.savefig(save_path, dpi=150, bbox_inches='tight')
                plt.show()
                print(f"✅ t-SNE visualization saved to {save_path}\n")

        # Early stopping
        if patience_counter >= early_stopping_patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

    # Load best model
    model.load_state_dict(best_model_state)

    print("="*70)
    print(f"TRAINING COMPLETE - Best Val F1: {best_val_f1:.4f}")
    print("="*70)

    return model, history


print("✅ Training functions defined (no reconstruction)")

In [ ]:
# ============================================================================
# TRAIN THE MODEL (with SCL + Prototype EMA)
# ============================================================================

print("Starting training with SCL + Prototype EMA...")
print(f"Configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"  SCL enabled: {USE_SCL} (λ_scl = {LAMBDA_SCL})")
print(f"  Prototype EMA enabled: {USE_PROTO_EMA} (momentum = {EMA_MOMENTUM})")
print()

trained_model, training_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lambda_class=LAMBDA_CLASS,
    lambda_kl=LAMBDA_KL,
    lambda_scl=LAMBDA_SCL,
    use_scl=USE_SCL,
    use_proto_ema=USE_PROTO_EMA,
    early_stopping_patience=EARLY_STOPPING_PATIENCE
)

# Save trained model
model_save_path = 'models/distribution_vae_best.pth'
torch.save({
    'model_state_dict': trained_model.state_dict(),
    'history': training_history,
    'hyperparameters': {
        'input_dim': INPUT_DIM,
        'latent_dim': LATENT_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_classes': NUM_CLASSES,
        'lambda_class': LAMBDA_CLASS,
        'lambda_kl': LAMBDA_KL,
        'lambda_scl': LAMBDA_SCL,
        'use_scl': USE_SCL,
        'use_proto_ema': USE_PROTO_EMA
    }
}, model_save_path)

print(f"\n✅ Model saved to {model_save_path}")


In [ ]:
# ============================================================================
# EVALUATE ON TEST SET (with CDAS)
# ============================================================================

from scipy.stats import spearmanr

def compute_CDAS(model, data_loader, device, num_classes=3):
    """
    Confidence-Distance Alignment Score (CDAS)

    Measures: Within each class, do high-confidence samples
              have smaller distances to their prototype?

    Components:
    - Correlation (40%): power 1.5 + p-value weighting
    - Spread (15%): IQR-based with adaptive threshold
    - Monotonicity (25%): weighted violation magnitude
    - Extreme Separation (20%): median-based, std-normalized

    Returns:
        overall_cdas: [0, 1], higher is better (>0.6 excellent)
        per_class_scores: Dictionary with detailed per-class metrics
    """
    model.eval()

    all_distances = []
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            mu, logvar, z, distances, scores = model(batch_x)

            all_distances.append(distances.cpu().numpy())
            all_scores.append(scores.cpu().numpy())
            all_labels.append(batch_y.numpy())

    all_distances = np.concatenate(all_distances)
    all_scores = np.concatenate(all_scores)
    all_labels = np.concatenate(all_labels)

    per_class_scores = {}

    for c in range(num_classes):
        mask = all_labels == c
        n_samples = mask.sum()

        if n_samples < 10:
            per_class_scores[c] = {
                'cdas': 0.0, 'correlation': 0.0, 'correlation_raw': 0.0,
                'p_value': 1.0, 'significance_weight': 0.0,
                'spread': 0.0, 'iqr': 0.0,
                'monotonicity': 0.0, 'extreme_sep': 0.0,
                'n_samples': n_samples
            }
            continue

        distances_c = all_distances[mask, c]
        confidences_c = all_scores[mask, c]

        # METRIC 1: Correlation (40%)
        corr, p_value = spearmanr(confidences_c, -distances_c)
        if np.isnan(corr):
            corr = 0.0
            p_value = 1.0

        significance_weight = 1.0 if p_value < 0.05 else 0.5
        corr_normalized = max(0, corr) ** 1.5
        corr_score = corr_normalized * significance_weight * 0.4

        # METRIC 2: Spread (15%)
        q75, q25 = np.percentile(confidences_c, [75, 25])
        iqr = q75 - q25
        confidence_range = confidences_c.max() - confidences_c.min()
        adaptive_threshold = max(0.15, confidence_range * 0.4)
        spread_normalized = min(1.0, iqr / adaptive_threshold)
        spread_score = spread_normalized * 0.15

        # METRIC 3: Monotonicity (25%)
        n_bins = 5
        bin_mean_dist = []

        try:
            bin_edges = np.percentile(confidences_c, np.linspace(0, 100, n_bins + 1))
            bin_edges[-1] += 1e-8
            bin_indices = np.digitize(confidences_c, bin_edges)

            for b in range(1, n_bins + 1):
                bin_mask = bin_indices == b
                if bin_mask.sum() > 0:
                    bin_mean_dist.append(distances_c[bin_mask].mean())

            if len(bin_mean_dist) > 1:
                dist_range = distances_c.max() - distances_c.min() + 1e-8
                weighted_violations = sum(
                    max(0, bin_mean_dist[i + 1] - bin_mean_dist[i]) / dist_range
                    for i in range(len(bin_mean_dist) - 1)
                )
                monotonicity_normalized = max(0, 1.0 - weighted_violations * 2)
            else:
                monotonicity_normalized = 0.5
        except:
            monotonicity_normalized = 0.5

        mono_score = monotonicity_normalized * 0.25

        # METRIC 4: Extreme Separation (20%)
        n_extreme = max(3, int(n_samples * 0.2))
        sorted_indices = np.argsort(confidences_c)
        top_idx = sorted_indices[-n_extreme:]
        bot_idx = sorted_indices[:n_extreme]

        top_dist = np.median(distances_c[top_idx])
        bot_dist = np.median(distances_c[bot_idx])
        separation = bot_dist - top_dist

        dist_std = distances_c.std() + 1e-8
        separation_effect = separation / (2 * dist_std)
        extreme_normalized = min(1.0, max(0, separation_effect))
        extreme_score = extreme_normalized * 0.20

        class_cdas = corr_score + spread_score + mono_score + extreme_score

        per_class_scores[c] = {
            'cdas': class_cdas,
            'correlation': corr_normalized,
            'correlation_raw': corr,
            'p_value': p_value,
            'significance_weight': significance_weight,
            'spread': spread_normalized,
            'iqr': iqr,
            'monotonicity': monotonicity_normalized,
            'extreme_sep': extreme_normalized,
            'n_samples': n_samples
        }

    total_samples = sum(v['n_samples'] for v in per_class_scores.values() if v['n_samples'] >= 10)

    if total_samples > 0:
        weighted_cdas = sum(
            v['cdas'] * v['n_samples']
            for v in per_class_scores.values()
            if v['n_samples'] >= 10
        ) / total_samples
    else:
        weighted_cdas = 0.0

    return weighted_cdas, per_class_scores


# ============================================================================
# Run Evaluation
# ============================================================================

print("\n" + "="*70)
print("TEST SET EVALUATION")
print("="*70)

test_losses, test_metrics, test_labels, test_predictions, test_distances, test_scores = evaluate(
    trained_model, test_loader, device,
    LAMBDA_CLASS, LAMBDA_KL
)

print(f"\nTest Results:")
print(f"  Loss: {test_losses['total']:.4f}")
print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
print(f"  F1 Score: {test_metrics['f1']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall: {test_metrics['recall']:.4f}")

# Per-class metrics
print(f"\nPer-Class Metrics:")
print(f"{'Class':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 50)


# Compute per-class metrics correctly using precision_recall_fscore_support
from sklearn.metrics import precision_recall_fscore_support

precisions, recalls, f1s, supports = precision_recall_fscore_support(
    test_labels, test_predictions, labels=range(NUM_CLASSES), average=None, zero_division=0
)

for i, class_name in enumerate(CLASS_NAMES):
    print(f"{class_name:<12} {precisions[i]:<12.4f} {recalls[i]:<12.4f} {f1s[i]:<12.4f}")


# ============================================================================
# CDAS Evaluation
# ============================================================================

print("\n" + "="*70)
print("CDAS (Confidence-Distance Alignment Score) EVALUATION")
print("="*70)

cdas_score, cdas_per_class = compute_CDAS(trained_model, test_loader, device, NUM_CLASSES)

# Determine CDAS quality
if cdas_score > 0.6:
    cdas_quality = "Excellent"
    cdas_emoji = "\u2713\u2713"
elif cdas_score > 0.4:
    cdas_quality = "Good"
    cdas_emoji = "\u2713"
elif cdas_score > 0.2:
    cdas_quality = "Moderate"
    cdas_emoji = "\u25b3"
else:
    cdas_quality = "Poor"
    cdas_emoji = "\u2717"

print(f"\nOverall CDAS: {cdas_score:.4f} ({cdas_quality}) {cdas_emoji}")
print(f"\nInterpretation:")
print(f"  > 0.6: Excellent - Strong confidence-distance alignment")
print(f"  > 0.4: Good - Reasonable alignment")
print(f"  > 0.2: Moderate - Some alignment")
print(f"  < 0.2: Poor - Weak alignment")

print(f"\nPer-Class CDAS Breakdown:")
print(f"{'Class':<12} {'CDAS':<10} {'Corr':<10} {'Spread':<10} {'Mono':<10} {'ExtSep':<10} {'N':<8}")
print("-" * 70)

for c in range(NUM_CLASSES):
    scores = cdas_per_class[c]
    class_name = CLASS_NAMES[c] if c < len(CLASS_NAMES) else f"Class {c}"
    print(f"{class_name:<12} {scores['cdas']:<10.4f} {scores['correlation']:<10.4f} "
          f"{scores['spread']:<10.4f} {scores['monotonicity']:<10.4f} "
          f"{scores['extreme_sep']:<10.4f} {scores['n_samples']:<8}")

print("\n" + "="*70)
print("CDAS Components Explanation:")
print("-" * 70)
print("  Correlation (40%): Spearman correlation between confidence and -distance")
print("  Spread (15%):      IQR of confidence scores (diversity)")
print("  Monotonicity (25%): Distance decreases as confidence increases")
print("  ExtSep (20%):      High-conf samples closer than low-conf samples")
print("="*70)


# Comprehensive Visualizations

This section provides 12 comprehensive visualizations to analyze the model:

1. **Distance Distribution Analysis** - How well are classes separated?
2. **Distance Matrix Heatmap** - Average distances between class pairs
3. **Score Calibration Curves** - Are predicted scores reliable?
4. **Latent Space Visualization** - t-SNE of learned representations
5. **Semantic Drift Analysis** - Distance trajectory for manipulations
6. **Distance vs Confidence** - Relationship between distance and confidence
7. **Ablation Study** - Effect of different loss components
8. **ROC Curves** - Distance-based classification performance
9. **Confusion Matrix** - Classification errors
10. **Training Dynamics** - Loss and metric curves over time
11. **Prototype Evolution** - How prototypes change during training
12. **Baseline Comparison** - vs traditional classifier


## 1. Distance Distribution Analysis

For each true class, visualize the distribution of distances to all prototypes.
- **Goal**: Samples should be close to their own class prototype, far from others
- **Good separation**: Clear peaks with minimal overlap


In [ ]:
# ============================================================================
# VISUALIZATION 1: Distance Distribution Analysis
# ============================================================================

def plot_distance_distributions(model, data_loader, device, save_path='plots/distance_distributions.png'):
    """
    Plot distance distributions for each true class to all prototypes
    """
    model.eval()

    # Collect all distances and labels
    all_distances = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            _, _, _, distances, _ = model(batch_x)
            all_distances.append(distances.cpu())
            all_labels.append(batch_y.cpu())

    all_distances = torch.cat(all_distances).numpy()  # (N, num_classes)
    all_labels = torch.cat(all_labels).numpy()  # (N,)

    # Create subplots (one per true class)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for true_class in range(NUM_CLASSES):
        ax = axes[true_class]

        # Get samples belonging to this class
        mask = all_labels == true_class
        class_distances = all_distances[mask]  # (n_samples, num_classes)

        # Plot distance distribution to each prototype
        for proto_class in range(NUM_CLASSES):
            distances_to_proto = class_distances[:, proto_class]

            label = f'→ {CLASS_NAMES[proto_class]}'
            color = CLASS_COLORS[proto_class]
            linestyle = '-' if proto_class == true_class else '--'
            linewidth = 2.5 if proto_class == true_class else 1.5

            ax.hist(distances_to_proto, bins=50, alpha=0.6, color=color,
                   label=label, edgecolor='black', linewidth=0.5)

        ax.set_title(f'True Class: {CLASS_NAMES[true_class]}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Distance', fontsize=12)
        ax.set_ylabel('Frequency', fontsize=12)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

        # Add statistics
        intra_dist = class_distances[:, true_class].mean()
        inter_dist = np.mean([class_distances[:, c].mean()
                             for c in range(NUM_CLASSES) if c != true_class])
        ax.text(0.95, 0.95, f'Intra: {intra_dist:.2f}\nInter: {inter_dist:.2f}',
               transform=ax.transAxes, fontsize=10,
               verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Distance distributions saved to {save_path}")


# Generate plot
plot_distance_distributions(trained_model, test_loader, device)


## 2. Distance Matrix Heatmap

Average distances between all class pairs.
- **Diagonal**: Intra-class distances (should be small)
- **Off-diagonal**: Inter-class distances (should be large)


In [ ]:
# ============================================================================
# VISUALIZATION 2: Distance Matrix Heatmap
# ============================================================================

def plot_distance_matrix(model, data_loader, device, save_path='plots/distance_matrix.png'):
    """
    Create a heatmap showing average distances between class pairs
    """
    model.eval()

    # Initialize distance matrix
    distance_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES))
    count_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES))

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.cpu().numpy()

            _, _, _, distances, _ = model(batch_x)
            distances = distances.cpu().numpy()

            # Accumulate distances
            for i, true_class in enumerate(batch_y):
                for proto_class in range(NUM_CLASSES):
                    distance_matrix[true_class, proto_class] += distances[i, proto_class]
                    count_matrix[true_class, proto_class] += 1

    # Average distances
    distance_matrix = distance_matrix / (count_matrix + 1e-8)

    # Create heatmap
    fig, ax = plt.subplots(figsize=(10, 8))

    im = ax.imshow(distance_matrix, cmap='YlOrRd', aspect='auto')

    # Set ticks
    ax.set_xticks(np.arange(NUM_CLASSES))
    ax.set_yticks(np.arange(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticklabels(CLASS_NAMES)

    # Rotate x labels
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # Add text annotations
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            text = ax.text(j, i, f'{distance_matrix[i, j]:.2f}',
                          ha="center", va="center", color="black" if distance_matrix[i, j] < distance_matrix.max()/2 else "white",
                          fontsize=14, fontweight='bold')

    ax.set_title('Average Distance Matrix\n(Row: True Class, Column: Distance to Prototype)',
                fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Prototype Class', fontsize=12)
    ax.set_ylabel('True Class', fontsize=12)

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Average Distance', rotation=270, labelpad=20, fontsize=12)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Distance matrix saved to {save_path}")

    # Print summary statistics
    print("\nDistance Matrix Summary:")
    print(f"  Mean intra-class distance: {np.diag(distance_matrix).mean():.4f}")
    print(f"  Mean inter-class distance: {(distance_matrix.sum() - np.diag(distance_matrix).sum()) / (NUM_CLASSES * (NUM_CLASSES - 1)):.4f}")


# Generate plot
plot_distance_matrix(trained_model, test_loader, device)


## 4. Latent Space Visualization

t-SNE projection of learned latent representations.
- **Color**: True class
- **Stars**: Class prototypes
- **Goal**: Clear clustering by class


In [ ]:
# ============================================================================
# VISUALIZATION 4: Latent Space Visualization
# ============================================================================

def plot_latent_space(model, data_loader, device, save_path='plots/latent_space_tsne.png'):
    """
    Visualize latent space using t-SNE

    IMPORTANT: Data points and prototypes must be transformed together
    in a single t-SNE to be in the same 2D space!
    """
    model.eval()

    # Collect latent representations
    all_z = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            outputs = model(batch_x, return_all=True)
            all_z.append(outputs['z'].cpu())
            all_labels.append(batch_y.cpu())

    all_z = torch.cat(all_z).numpy()
    all_labels = torch.cat(all_labels).numpy()

    # Get prototypes
    prototypes = model.prototype_means.detach().cpu().numpy()

    # CRITICAL FIX: Combine data and prototypes BEFORE t-SNE
    # This ensures they are in the same 2D embedding space
    print("Computing t-SNE projection...")
    print(f"  Data points: {all_z.shape[0]}")
    print(f"  Prototypes: {prototypes.shape[0]}")

    # Stack data and prototypes together
    combined = np.vstack([all_z, prototypes])
    print(f"  Combined: {combined.shape}")

    # Single t-SNE transformation for everything
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
    combined_tsne = tsne.fit_transform(combined)

    # Split back into data and prototypes
    n_data = all_z.shape[0]
    z_tsne = combined_tsne[:n_data]  # Data points
    proto_tsne = combined_tsne[n_data:]  # Prototypes (last 3)

    print(f"✅ t-SNE complete")
    print(f"  Data t-SNE: {z_tsne.shape}")
    print(f"  Prototype t-SNE: {proto_tsne.shape}")

    # Plot
    fig, ax = plt.subplots(figsize=(12, 10))

    # Plot samples
    for class_idx in range(NUM_CLASSES):
        mask = all_labels == class_idx
        ax.scatter(z_tsne[mask, 0], z_tsne[mask, 1],
                  c=CLASS_COLORS[class_idx], label=CLASS_NAMES[class_idx],
                  alpha=0.6, s=30, edgecolors='black', linewidths=0.5)

    # Plot prototypes
    for class_idx in range(NUM_CLASSES):
        ax.scatter(proto_tsne[class_idx, 0], proto_tsne[class_idx, 1],
                  c=CLASS_COLORS[class_idx], marker='*', s=800,
                  edgecolors='black', linewidths=2,
                  label=f'{CLASS_NAMES[class_idx]} Prototype', zorder=10)

    ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
    ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
    ax.set_title('Latent Space Visualization (t-SNE)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Latent space visualization saved to {save_path}")


# Generate plot
plot_latent_space(trained_model, test_loader, device)

In [ ]:
# ============================================================================
# VISUALIZATION 7: ROC Curves (Distance-based)
# ============================================================================

def plot_roc_curves(test_distances, test_labels, save_path='plots/roc_curves.png'):
    """
    ROC curves for each class using distance thresholding
    """
    test_distances = test_distances.numpy()
    test_labels = test_labels.numpy()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for class_idx in range(NUM_CLASSES):
        ax = axes[class_idx]

        # Binary labels
        binary_labels = (test_labels == class_idx).astype(int)

        # Use negative distance as score (lower distance = higher score)
        scores = -test_distances[:, class_idx]

        # Compute ROC curve
        fpr, tpr, thresholds = roc_curve(binary_labels, scores)
        roc_auc = auc(fpr, tpr)

        # Plot
        ax.plot(fpr, tpr, color=CLASS_COLORS[class_idx], linewidth=2.5,
               label=f'{CLASS_NAMES[class_idx]} (AUC = {roc_auc:.3f})')
        ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random')

        ax.set_xlabel('False Positive Rate', fontsize=12)
        ax.set_ylabel('True Positive Rate', fontsize=12)
        ax.set_title(f'{CLASS_NAMES[class_idx]} ROC Curve', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ ROC curves saved to {save_path}")


plot_roc_curves(test_distances, test_labels)


In [ ]:
# ============================================================================
# VISUALIZATION 8: Confusion Matrix
# ============================================================================

def plot_confusion_matrix_dist(test_predictions, test_labels, save_path='plots/confusion_matrix.png'):
    """
    Confusion matrix for distance-based predictions
    """
    test_predictions = test_predictions.numpy()
    test_labels = test_labels.numpy()

    cm = confusion_matrix(test_labels, test_predictions)

    # Normalize
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(10, 8))

    im = ax.imshow(cm_norm, cmap='Blues', aspect='auto', vmin=0, vmax=1)

    # Set ticks
    ax.set_xticks(np.arange(NUM_CLASSES))
    ax.set_yticks(np.arange(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticklabels(CLASS_NAMES)

    # Rotate labels
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # Add text annotations
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            text = ax.text(j, i, f'{cm[i, j]}\n({cm_norm[i, j]:.2%})',
                          ha="center", va="center",
                          color="white" if cm_norm[i, j] > 0.5 else "black",
                          fontsize=12, fontweight='bold')

    ax.set_title('Confusion Matrix (Distance-Based Predictions)',
                fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Predicted Class', fontsize=12)
    ax.set_ylabel('True Class', fontsize=12)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Normalized Frequency', rotation=270, labelpad=20, fontsize=12)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Confusion matrix saved to {save_path}")


plot_confusion_matrix_dist(test_predictions, test_labels)


In [ ]:
# ============================================================================
# VISUALIZATION 9: Training Dynamics
# ============================================================================

def plot_training_dynamics(history, save_path='plots/training_dynamics.png'):
    """
    Multi-panel plot showing training dynamics
    """
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Panel 1: Loss curves
    ax = axes[0, 0]
    ax.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
    ax.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val Loss')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('Loss Curves', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    # Panel 2: F1 Score
    ax = axes[0, 1]
    ax.plot(epochs, history['train_f1'], 'b-', linewidth=2, label='Train F1')
    ax.plot(epochs, history['val_f1'], 'r-', linewidth=2, label='Val F1')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title('F1 Score Over Time', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    # Panel 3: Accuracy
    ax = axes[1, 0]
    ax.plot(epochs, history['val_accuracy'], 'g-', linewidth=2, label='Val Accuracy')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title('Validation Accuracy', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    # Panel 4: Learning Rate
    ax = axes[1, 1]
    ax.plot(epochs, history['learning_rates'], 'm-', linewidth=2, label='Learning Rate')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Learning Rate', fontsize=12)
    ax.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    ax.set_yscale('log')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Training dynamics saved to {save_path}")


plot_training_dynamics(training_history)


## Summary

All visualizations have been generated and saved to the `plots/` directory.

### Key Findings

Check the visualizations for:
1. **Separation Quality**: Are classes well-separated in distance space?
2. **Calibration**: Are scores well-calibrated?
3. **Latent Structure**: Is the latent space well-organized?
4. **Performance**: How does this compare to the classifier baseline?

### Next Steps

1. Analyze semantic drift on manipulated images
2. Compare with classifier baseline quantitatively
3. Run ablation studies with different loss weights
4. Fine-tune hyperparameters based on visualizations


## 10. Semantic Drift Analysis (Optional)

If you have manipulated images, this visualization shows how distances change.

**Note**: This requires pairs of (original, manipulated) images.
Skip this if you don't have manipulated data.


## 11. Ablation Study

Compare model performance with different loss configurations.

This requires re-training the model with different loss weights.


In [ ]:
# ============================================================================
# VISUALIZATION 11: Ablation Study (All 15 Combinations + CDAS)
# ============================================================================
#
# Full ablation study testing ALL possible combinations of 4 components.
# Note: Reconstruction loss removed (decoder not used in this approach)
#
# Components (3 total):
# - L_scl: Supervised Contrastive Loss
# - Prototype EMA: EMA-based prototype updates
# - L_kl: KL divergence
#
# Metrics:
# - F1, Accuracy, Precision, Recall (Classification)
# - Separation Ratio, Silhouette Score (Distribution)
# - D-Corr, C-Corr (Correlation)
# - CDAS (Confidence-Distance Alignment Score)
#
# Total combinations: 2^3 - 1 = 7 (excluding all-off which is invalid)
# ============================================================================

from itertools import product
from sklearn.metrics import silhouette_score
from scipy.stats import entropy, spearmanr

# ============================================================================
# CDAS (Confidence-Distance Alignment Score)
# ============================================================================

def compute_CDAS(model, data_loader, device, num_classes=3):
    """
    Confidence-Distance Alignment Score (CDAS)

    Measures: Within each class, do high-confidence samples
              have smaller distances to their prototype?

    Components:
    - Correlation (40%): power 1.5 + p-value weighting
    - Spread (15%): IQR-based with adaptive threshold
    - Monotonicity (25%): weighted violation magnitude
    - Extreme Separation (20%): median-based, std-normalized

    Returns:
        overall_cdas: [0, 1], higher is better (>0.6 excellent)
        per_class_scores: Dictionary with detailed per-class metrics
    """
    model.eval()

    all_distances = []
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            mu, logvar, z, distances, scores = model(batch_x)

            all_distances.append(distances.cpu().numpy())
            all_scores.append(scores.cpu().numpy())
            all_labels.append(batch_y.numpy())

    all_distances = np.concatenate(all_distances)
    all_scores = np.concatenate(all_scores)
    all_labels = np.concatenate(all_labels)

    per_class_scores = {}

    for c in range(num_classes):
        mask = all_labels == c
        n_samples = mask.sum()

        if n_samples < 10:
            per_class_scores[c] = {
                'cdas': 0.0, 'correlation': 0.0, 'correlation_raw': 0.0,
                'p_value': 1.0, 'significance_weight': 0.0,
                'spread': 0.0, 'iqr': 0.0,
                'monotonicity': 0.0, 'extreme_sep': 0.0,
                'n_samples': n_samples, 'bin_mean_conf': [], 'bin_mean_dist': []
            }
            continue

        distances_c = all_distances[mask, c]
        confidences_c = all_scores[mask, c]

        # METRIC 1: Correlation (40%)
        corr, p_value = spearmanr(confidences_c, -distances_c)
        if np.isnan(corr):
            corr = 0.0
            p_value = 1.0

        significance_weight = 1.0 if p_value < 0.05 else 0.5
        corr_normalized = max(0, corr) ** 1.5
        corr_score = corr_normalized * significance_weight * 0.4

        # METRIC 2: Spread (15%)
        q75, q25 = np.percentile(confidences_c, [75, 25])
        iqr = q75 - q25
        confidence_range = confidences_c.max() - confidences_c.min()
        adaptive_threshold = max(0.15, confidence_range * 0.4)
        spread_normalized = min(1.0, iqr / adaptive_threshold)
        spread_score = spread_normalized * 0.15

        # METRIC 3: Monotonicity (25%)
        n_bins = 5
        bin_mean_dist = []
        bin_mean_conf = []

        try:
            bin_edges = np.percentile(confidences_c, np.linspace(0, 100, n_bins + 1))
            bin_edges[-1] += 1e-8
            bin_indices = np.digitize(confidences_c, bin_edges)

            for b in range(1, n_bins + 1):
                bin_mask = bin_indices == b
                if bin_mask.sum() > 0:
                    bin_mean_dist.append(distances_c[bin_mask].mean())
                    bin_mean_conf.append(confidences_c[bin_mask].mean())

            if len(bin_mean_dist) > 1:
                dist_range = distances_c.max() - distances_c.min() + 1e-8
                weighted_violations = sum(
                    max(0, bin_mean_dist[i + 1] - bin_mean_dist[i]) / dist_range
                    for i in range(len(bin_mean_dist) - 1)
                )
                monotonicity_normalized = max(0, 1.0 - weighted_violations * 2)
            else:
                monotonicity_normalized = 0.5
        except:
            monotonicity_normalized = 0.5

        mono_score = monotonicity_normalized * 0.25

        # METRIC 4: Extreme Separation (20%)
        n_extreme = max(3, int(n_samples * 0.2))
        sorted_indices = np.argsort(confidences_c)
        top_idx = sorted_indices[-n_extreme:]
        bot_idx = sorted_indices[:n_extreme]

        top_dist = np.median(distances_c[top_idx])
        bot_dist = np.median(distances_c[bot_idx])
        separation = bot_dist - top_dist

        dist_std = distances_c.std() + 1e-8
        separation_effect = separation / (2 * dist_std)
        extreme_normalized = min(1.0, max(0, separation_effect))
        extreme_score = extreme_normalized * 0.20

        class_cdas = corr_score + spread_score + mono_score + extreme_score

        per_class_scores[c] = {
            'cdas': class_cdas,
            'correlation': corr_normalized,
            'correlation_raw': corr,
            'p_value': p_value,
            'significance_weight': significance_weight,
            'spread': spread_normalized,
            'iqr': iqr,
            'monotonicity': monotonicity_normalized,
            'extreme_sep': extreme_normalized,
            'n_samples': n_samples,
            'bin_mean_conf': bin_mean_conf,
            'bin_mean_dist': bin_mean_dist
        }

    total_samples = sum(v['n_samples'] for v in per_class_scores.values() if v['n_samples'] >= 10)

    if total_samples > 0:
        weighted_cdas = sum(
            v['cdas'] * v['n_samples']
            for v in per_class_scores.values()
            if v['n_samples'] >= 10
        ) / total_samples
    else:
        weighted_cdas = 0.0

    return weighted_cdas, per_class_scores


def generate_all_configurations():
    """
    Generate all 15 valid configurations (2^3 - 1).
    3 components: SCL, EMA, KL (no reconstruction, no dist)
    """
    configurations = []

    # Generate all 2^4 = 16 combinations
    for flags in product([False, True], repeat=3):
        scl, ema, kl = flags

        # Skip the all-off configuration
        if not any(flags):
            continue

        # Create configuration name
        active = []
        if scl: active.append('SCL')
        if ema: active.append('EMA')
        if kl: active.append('KL')

        name = '+'.join(active) if active else 'None'

        config = {
            'name': name,
            'lambda_class': LAMBDA_CLASS,  # Always on
            'lambda_kl': LAMBDA_KL if kl else 0.0,
            'lambda_scl': LAMBDA_SCL if scl else 0.0,
            'use_scl': scl,
            'use_proto_ema': ema,
            'active_components': {
                'scl': scl, 'ema': ema, 'kl': kl
            }
        }
        configurations.append(config)

    return configurations


def run_full_ablation_study(train_loader, val_loader, test_loader, device, num_epochs=30):
    """
    Full ablation study testing all 15 combinations of 4 components.

    Components:
    - L_scl: Supervised Contrastive Loss
    - Prototype EMA: EMA-based prototype updates
    - L_kl: KL divergence

    Note: No reconstruction loss (decoder removed)
    """

    configurations = generate_all_configurations()
    results = {}

    print("\n" + "="*80)
    print("FULL ABLATION STUDY (All 15 Combinations)")
    print("="*80)
    print(f"Components: SCL, Prototype EMA, L_kl")
    print(f"Total configurations: {len(configurations)} (2^3 - 1)")
    print(f"Epochs per config: {num_epochs}")
    print(f"Note: Reconstruction loss removed (no decoder)")
    print("="*80)

    for idx, config in enumerate(configurations, 1):
        print(f"\n[{idx}/{len(configurations)}] {config['name']}")
        active = config['active_components']
        print(f"  Components: SCL={active['scl']}, EMA={active['ema']}, "
              f"KL={active['kl']}")

        # Initialize fresh model (PrototypeEncoder, no decoder)
        model = PrototypeEncoder(
            input_dim=INPUT_DIM,
            latent_dim=LATENT_DIM,
            hidden_dim=HIDDEN_DIM,
            num_classes=NUM_CLASSES
        ).to(device)

        prototype_ema.reset()

        # Train
        trained_model, history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=num_epochs,
            learning_rate=LEARNING_RATE,
            lambda_class=config['lambda_class'],
            lambda_kl=config['lambda_kl'],
            lambda_scl=config['lambda_scl'],
            use_scl=config['use_scl'],
            use_proto_ema=config['use_proto_ema'],
            early_stopping_patience=10
        )

        # Evaluate
        _, metrics, test_labels, test_preds, test_distances, test_scores = evaluate(
            trained_model, test_loader, device,
            config['lambda_class'],
            config['lambda_kl'], config['lambda_scl']
        )

        # Distribution metrics
        dist_metrics = compute_distribution_metrics_for_ablation(
            trained_model, test_loader, device
        )

        # CDAS
        cdas_score, cdas_per_class = compute_CDAS(
            trained_model, test_loader, device, NUM_CLASSES
        )

        # Correlation metrics
        test_distances_np = test_distances.numpy()
        test_labels_np = test_labels.numpy()
        test_scores_np = test_scores.numpy()

        min_distances = test_distances_np.min(axis=1)
        predictions = test_distances_np.argmin(axis=1)
        confidences = test_scores_np.max(axis=1)
        correct = (predictions == test_labels_np).astype(float)

        dist_corr, _ = spearmanr(min_distances, correct)
        conf_corr, _ = spearmanr(confidences, correct)

        results[config['name']] = {
            'f1': metrics['f1'],
            'accuracy': metrics['accuracy'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'separation_ratio': dist_metrics['separation_ratio'],
            'silhouette_score': dist_metrics['silhouette_score'],
            'dist_correct_corr': dist_corr if not np.isnan(dist_corr) else 0.0,
            'conf_correct_corr': conf_corr if not np.isnan(conf_corr) else 0.0,
            'cdas': cdas_score,
            'cdas_per_class': cdas_per_class,
            'config': config,
            'num_components': sum(config['active_components'].values())
        }

        print(f"  ✓ F1: {metrics['f1']:.4f} | Acc: {metrics['accuracy']:.4f}")
        print(f"  ✓ Sep.Ratio: {dist_metrics['separation_ratio']:.2f} | Silhouette: {dist_metrics['silhouette_score']:.4f}")

        cdas_status = '✓✓' if cdas_score > 0.6 else '✓' if cdas_score > 0.4 else '△' if cdas_score > 0.2 else '✗'
        print(f"  ✓ CDAS: {cdas_score:.4f} {cdas_status}")

    visualize_full_ablation_results(results)

    return results


def visualize_full_ablation_results(results):
    """Visualize results for 15 configurations."""
    print("\n" + "="*80)
    print("Creating visualizations for all 15 configurations...")
    print("="*80)

    fig = plt.figure(figsize=(24, 20))

    names = list(results.keys())
    sorted_names = sorted(names, key=lambda n: results[n]['f1'], reverse=True)
    sorted_by_cdas = sorted(names, key=lambda n: results[n]['cdas'], reverse=True)

    # Plot 1: F1 Score Ranking
    ax1 = fig.add_subplot(3, 3, 1)
    f1_scores = [results[n]['f1'] for n in sorted_names]
    colors = plt.cm.RdYlGn([s/max(f1_scores) for s in f1_scores])

    ax1.barh(range(len(sorted_names)), f1_scores, color=colors)
    ax1.set_yticks(range(len(sorted_names)))
    ax1.set_yticklabels(sorted_names, fontsize=9)
    ax1.set_xlabel('F1 Score', fontsize=11)
    ax1.set_title('All 15 Configurations by F1', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    ax1.invert_yaxis()

    for i, v in enumerate(f1_scores):
        ax1.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)

    # Plot 2: CDAS Ranking
    ax2 = fig.add_subplot(3, 3, 2)
    cdas_scores = [results[n]['cdas'] for n in sorted_by_cdas]

    colors = ['green' if s > 0.6 else 'yellowgreen' if s > 0.4 else 'gold' if s > 0.2 else 'salmon' for s in cdas_scores]

    ax2.barh(range(len(sorted_by_cdas)), cdas_scores, color=colors, alpha=0.8)
    ax2.set_yticks(range(len(sorted_by_cdas)))
    ax2.set_yticklabels(sorted_by_cdas, fontsize=9)
    ax2.set_xlabel('CDAS Score', fontsize=11)
    ax2.set_title('All 15 Configurations by CDAS', fontsize=12, fontweight='bold')
    ax2.axvline(0.6, color='green', linestyle='--', alpha=0.5, label='Excellent')
    ax2.axvline(0.4, color='orange', linestyle='--', alpha=0.5, label='Good')
    ax2.set_xlim([0, 1])
    ax2.grid(True, alpha=0.3, axis='x')
    ax2.invert_yaxis()
    ax2.legend(fontsize=8, loc='lower right')

    for i, v in enumerate(cdas_scores):
        ax2.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=8)

    # Plot 3: Component Frequency in Top 5
    ax3 = fig.add_subplot(3, 3, 3)

    top_5 = sorted_names[:5]
    comp_counts = {'SCL': 0, 'EMA': 0, 'KL': 0}

    for name in top_5:
        active = results[name]['config']['active_components']
        if active['scl']: comp_counts['SCL'] += 1
        if active['ema']: comp_counts['EMA'] += 1
        if active['kl']: comp_counts['KL'] += 1

    comps = list(comp_counts.keys())
    counts = list(comp_counts.values())
    colors = plt.cm.Set2(range(4))

    bars = ax3.bar(comps, counts, color=colors)
    ax3.set_ylabel('Frequency in Top 5', fontsize=11)
    ax3.set_title('Component Frequency (Top 5 by F1)', fontsize=12, fontweight='bold')
    ax3.set_ylim([0, 5])
    ax3.grid(True, alpha=0.3, axis='y')

    for bar, count in zip(bars, counts):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(count), ha='center', fontsize=11, fontweight='bold')

    # Plot 4: F1 vs CDAS
    ax4 = fig.add_subplot(3, 3, 4)

    f1_all = [results[n]['f1'] for n in names]
    cdas_all = [results[n]['cdas'] for n in names]
    num_comps = [results[n]['num_components'] for n in names]

    scatter = ax4.scatter(f1_all, cdas_all, c=num_comps, cmap='viridis',
                          s=100, alpha=0.7, edgecolors='black', linewidths=0.5)
    plt.colorbar(scatter, ax=ax4, label='# Components')

    # Annotate all points
    for i, name in enumerate(names):
        ax4.annotate(name, (f1_all[i], cdas_all[i]), fontsize=7, alpha=0.8)

    ax4.axhline(0.4, color='orange', linestyle='--', alpha=0.3)
    ax4.set_xlabel('F1 Score', fontsize=11)
    ax4.set_ylabel('CDAS Score', fontsize=11)
    ax4.set_title('F1 vs CDAS (color=# components)', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3)

    # Plot 5: Component Impact
    ax5 = fig.add_subplot(3, 3, 5)

    components = ['SCL', 'EMA', 'KL']
    comp_keys = ['scl', 'ema', 'kl']

    cdas_impact = []
    f1_impact = []

    for key in comp_keys:
        on_cdas = [results[n]['cdas'] for n in names if results[n]['config']['active_components'][key]]
        off_cdas = [results[n]['cdas'] for n in names if not results[n]['config']['active_components'][key]]
        on_f1 = [results[n]['f1'] for n in names if results[n]['config']['active_components'][key]]
        off_f1 = [results[n]['f1'] for n in names if not results[n]['config']['active_components'][key]]

        cdas_impact.append(np.mean(on_cdas) - np.mean(off_cdas) if off_cdas else 0)
        f1_impact.append(np.mean(on_f1) - np.mean(off_f1) if off_f1 else 0)

    x = np.arange(len(components))
    width = 0.35

    ax5.bar(x - width/2, f1_impact, width, label='F1 Impact', color='steelblue')
    ax5.bar(x + width/2, cdas_impact, width, label='CDAS Impact', color='coral')

    ax5.axhline(0, color='black', linestyle='-', linewidth=0.5)
    ax5.set_xlabel('Component', fontsize=11)
    ax5.set_ylabel('Impact (ON - OFF)', fontsize=11)
    ax5.set_title('Component Impact on F1 & CDAS', fontsize=12, fontweight='bold')
    ax5.set_xticks(x)
    ax5.set_xticklabels(components)
    ax5.legend()
    ax5.grid(True, alpha=0.3, axis='y')

    # Plot 6: Sep Ratio vs Silhouette
    ax6 = fig.add_subplot(3, 3, 6)

    sep_all = [results[n]['separation_ratio'] for n in names]
    sil_all = [results[n]['silhouette_score'] for n in names]

    scatter = ax6.scatter(sep_all, sil_all, c=cdas_all, cmap='RdYlGn',
                          s=100, alpha=0.7, edgecolors='black', linewidths=0.5, vmin=0, vmax=0.8)
    plt.colorbar(scatter, ax=ax6, label='CDAS Score')

    ax6.set_xlabel('Separation Ratio', fontsize=11)
    ax6.set_ylabel('Silhouette Score', fontsize=11)
    ax6.set_title('Distribution Metrics (color=CDAS)', fontsize=12, fontweight='bold')
    ax6.grid(True, alpha=0.3)

    # Plot 7: Performance by # Components
    ax7 = fig.add_subplot(3, 3, 7)

    for n_comp in range(1, 5):
        mask = [nc == n_comp for nc in num_comps]
        f1_group = [f for f, m in zip(f1_all, mask) if m]
        x_group = [n_comp + np.random.uniform(-0.15, 0.15) for _ in f1_group]

        ax7.scatter(x_group, f1_group, alpha=0.7, s=80, label=f'{n_comp} comp')

    mean_f1 = {}
    for n_comp in range(1, 5):
        mask = [nc == n_comp for nc in num_comps]
        f1_group = [f for f, m in zip(f1_all, mask) if m]
        if f1_group:
            mean_f1[n_comp] = np.mean(f1_group)

    ax7.plot(list(mean_f1.keys()), list(mean_f1.values()),
             'r-o', linewidth=2, markersize=10, label='Mean F1')

    ax7.set_xlabel('Number of Components', fontsize=11)
    ax7.set_ylabel('F1 Score', fontsize=11)
    ax7.set_title('F1 vs # Components', fontsize=12, fontweight='bold')
    ax7.set_xticks([1, 2, 3, 4])
    ax7.legend(fontsize=8)
    ax7.grid(True, alpha=0.3)

    # Plot 8: CDAS by # Components
    ax8 = fig.add_subplot(3, 3, 8)

    mean_cdas = {}
    for n_comp in range(1, 5):
        mask = [nc == n_comp for nc in num_comps]
        cdas_group = [c for c, m in zip(cdas_all, mask) if m]
        x_group = [n_comp + np.random.uniform(-0.15, 0.15) for _ in cdas_group]

        ax8.scatter(x_group, cdas_group, alpha=0.7, s=80)

        if cdas_group:
            mean_cdas[n_comp] = np.mean(cdas_group)

    ax8.plot(list(mean_cdas.keys()), list(mean_cdas.values()),
             'r-o', linewidth=2, markersize=10, label='Mean CDAS')

    ax8.axhline(0.6, color='green', linestyle='--', alpha=0.5, label='Excellent')
    ax8.axhline(0.4, color='orange', linestyle='--', alpha=0.5, label='Good')

    ax8.set_xlabel('Number of Components', fontsize=11)
    ax8.set_ylabel('CDAS Score', fontsize=11)
    ax8.set_title('CDAS vs # Components', fontsize=12, fontweight='bold')
    ax8.set_xticks([1, 2, 3, 4])
    ax8.legend(fontsize=8)
    ax8.grid(True, alpha=0.3)

    # Plot 9: Summary
    ax9 = fig.add_subplot(3, 3, 9)
    ax9.axis('off')

    best_f1_name = sorted_names[0]
    best_cdas_name = sorted_by_cdas[0]
    combined_scores = {n: (results[n]['f1'] + results[n]['cdas'])/2 for n in names}
    best_combined_name = max(combined_scores, key=combined_scores.get)

    summary_text = f"""
╔════════════════════════════════════════════════════════════════╗
║           ABLATION STUDY SUMMARY (15 Configs)                  ║
║                    No Reconstruction Loss                      ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  🏆 BEST BY F1: {best_f1_name:<30}          ║
║     F1: {results[best_f1_name]['f1']:.4f}  CDAS: {results[best_f1_name]['cdas']:.4f}                       ║
║                                                                ║
║  🎯 BEST BY CDAS: {best_cdas_name:<28}          ║
║     F1: {results[best_cdas_name]['f1']:.4f}  CDAS: {results[best_cdas_name]['cdas']:.4f}                       ║
║                                                                ║
║  ⭐ BEST COMBINED: {best_combined_name:<27}          ║
║     F1: {results[best_combined_name]['f1']:.4f}  CDAS: {results[best_combined_name]['cdas']:.4f}                       ║
║                                                                ║
╠════════════════════════════════════════════════════════════════╣
║  📊 F1 Statistics:                                             ║
║     Mean: {np.mean(f1_all):.4f}  Std: {np.std(f1_all):.4f}                      ║
║     Max:  {max(f1_all):.4f}  Min: {min(f1_all):.4f}                      ║
║                                                                ║
║  📊 CDAS Statistics:                                           ║
║     Mean: {np.mean(cdas_all):.4f}  Std: {np.std(cdas_all):.4f}                      ║
║     Max:  {max(cdas_all):.4f}  Min: {min(cdas_all):.4f}                      ║
║                                                                ║
╠════════════════════════════════════════════════════════════════╣
║  📈 COMPONENT IMPACT ON CDAS:                                  ║
║     SCL:  {cdas_impact[0]:+.4f} {'✓✓' if cdas_impact[0] > 0.03 else '✓' if cdas_impact[0] > 0 else '✗'}                                   ║
║     EMA:  {cdas_impact[1]:+.4f} {'✓✓' if cdas_impact[1] > 0.03 else '✓' if cdas_impact[1] > 0 else '✗'}                                   ║
║     KL:   {cdas_impact[2]:+.4f} {'✓✓' if cdas_impact[2] > 0.03 else '✓' if cdas_impact[2] > 0 else '✗'}                                   ║
╚════════════════════════════════════════════════════════════════╝
    """

    ax9.text(0.02, 0.98, summary_text, transform=ax9.transAxes,
             fontsize=9, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.savefig('plots/ablation_study_15_configs.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Print results table
    print("\n" + "="*100)
    print("FULL ABLATION RESULTS (Sorted by F1 Score)")
    print("="*100)
    print(f"{'Rank':<5} {'Configuration':<20} {'F1':>8} {'Acc':>8} {'Sep.R':>8} {'Silh.':>8} {'CDAS':>8} {'#Comp':>6}")
    print("-"*100)

    for rank, name in enumerate(sorted_names, 1):
        r = results[name]
        cdas_mark = '✓✓' if r['cdas'] > 0.6 else '✓' if r['cdas'] > 0.4 else '△' if r['cdas'] > 0.2 else '✗'
        print(f"{rank:<5} {name:<20} {r['f1']:>8.4f} {r['accuracy']:>8.4f} "
              f"{r['separation_ratio']:>8.2f} {r['silhouette_score']:>8.4f} "
              f"{r['cdas']:>7.4f}{cdas_mark} {r['num_components']:>6}")

    print("-"*100)
    print("\nResults saved to: plots/ablation_study_15_configs.png")


def compute_distribution_metrics_for_ablation(model, data_loader, device):
    """Compute distribution-based metrics for ablation study."""
    model.eval()

    all_latents = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            mu, logvar, z, distances, scores = model(batch_x)
            all_latents.append(z.cpu().numpy())
            all_labels.append(batch_y.numpy())

    all_latents = np.concatenate(all_latents, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Intra-class distance
    intra_class_dists = []
    for c in range(NUM_CLASSES):
        mask = all_labels == c
        if mask.sum() > 1:
            class_latents = all_latents[mask]
            centroid = class_latents.mean(axis=0)
            dists = np.linalg.norm(class_latents - centroid, axis=1)
            intra_class_dists.append(dists.mean())
    intra_class_dist = np.mean(intra_class_dists) if intra_class_dists else 0

    # Inter-class distance
    centroids = []
    for c in range(NUM_CLASSES):
        mask = all_labels == c
        if mask.sum() > 0:
            centroids.append(all_latents[mask].mean(axis=0))

    inter_class_dists = []
    for i in range(len(centroids)):
        for j in range(i+1, len(centroids)):
            inter_class_dists.append(np.linalg.norm(centroids[i] - centroids[j]))
    inter_class_dist = np.mean(inter_class_dists) if inter_class_dists else 0

    separation_ratio = inter_class_dist / (intra_class_dist + 1e-8)

    # Silhouette score
    if len(np.unique(all_labels)) > 1:
        try:
            sil_score = silhouette_score(all_latents, all_labels)
        except:
            sil_score = 0.0
    else:
        sil_score = 0.0

    return {
        'intra_class_dist': intra_class_dist,
        'inter_class_dist': inter_class_dist,
        'separation_ratio': separation_ratio,
        'silhouette_score': sil_score
    }


# ============================================================================
# RUN ABLATION STUDY
# ============================================================================

RUN_ABLATION = True  # Set to True to run

if RUN_ABLATION:
    print("\n" + "="*80)
    print("Starting FULL Ablation Study (15 Configurations)")
    print("Components: SCL, Prototype EMA, L_kl")
    print("Note: Reconstruction loss removed (no decoder)")
    print("="*80)
    print("⚠️  WARNING: This will train 15 models!")
    print(f"Estimated time: ~5-7 hours (15 configs × 30 epochs)")
    print("="*80)

    ablation_results = run_full_ablation_study(
        train_loader, val_loader, test_loader,
        device, num_epochs=30
    )

    import pickle
    with open('ablation_results_15_configs.pkl', 'wb') as f:
        pickle.dump(ablation_results, f)
    print("\nResults saved to: ablation_results_15_configs.pkl")

else:
    print("\n" + "="*80)
    print("FULL ABLATION STUDY (All 15 Combinations)")
    print("="*80)
    print("💡 Ablation study is DISABLED")
    print("To run: Set RUN_ABLATION = True")
    print("\n📋 Components to test (4 total):")
    print("   • L_scl (Supervised Contrastive Loss)")
    print("   • Prototype EMA")
    print("   • L_kl (KL Divergence)")
    print("\n⚠️  Reconstruction loss removed (no decoder)")
    print("\n🔢 Total configurations: 2^3 - 1 = 7")
    print("⏱️  Estimated time: ~5-7 hours")

    configs = generate_all_configurations()
    print("\n" + "-"*50)
    print("All 15 configurations:")
    print("-"*50)
    for i, cfg in enumerate(configs, 1):
        active = cfg['active_components']
        flags = [k.upper() for k, v in active.items() if v]
        print(f"  {i:2d}. {cfg['name']:<20} ({len(flags)} components)")


# Semantic Drift Visualization

## Compare Original vs Manipulated Images

This section provides tools to visualize how an image's semantic representation changes after manipulation.

### Features

1. **Latent Space Visualization**: Shows both images' positions in t-SNE projected latent space
   - Background: Training data colored by class
   - Stars: Class prototypes
   - Arrow: Drift direction from original to manipulated

2. **Class Probability Comparison**: Bar chart comparing probability scores
   - Blue: Original image scores
   - Red: Manipulated image scores
   - Arrows: Score changes (↑ increase, ↓ decrease)

3. **Distance to Prototypes**: Mahalanobis distances to each class prototype

### Usage

```python
result = visualize_semantic_drift(
    original_path='path/to/original.jpg',
    manipulated_path='path/to/manipulated.jpg',
    model=trained_model,
    clip_model=clip_model,
    clip_processor=clip_processor,
    data_loader=test_loader,
    device=device
)
```

### Output

- Visualization saved to `plots/semantic_drift_comparison.png`
- Returns dictionary with detailed metrics


In [ ]:
# ============================================================================
# ABLATION STUDY: KL Divergence + Classifier Loss Only
# ============================================================================

print("=" * 80)
print("ABLATION STUDY: KL + Classifier Loss Only")
print("=" * 80)

# ============================================================================
# Load Test Images
# ============================================================================

import os
from PIL import Image
import glob

# Define paths for Colab environment
violence_low_dir = '/content/drive/MyDrive/vdsh_watermarking/inference_test/violence_low'
violence_high_dir = '/content/drive/MyDrive/vdsh_watermarking/inference_test/violence_high'

# Load test images
def load_test_images(low_dir, high_dir, num_images=2):
    """Load test images from the specified directories"""

    print(f"\nSearching for images in:")
    print(f"  Low violence: {low_dir}")
    print(f"  High violence: {high_dir}")

    # Get all image files
    low_images = []
    high_images = []

    if os.path.exists(low_dir):
        low_patterns = [
            os.path.join(low_dir, '*.jpg'),
            os.path.join(low_dir, '*.jpeg'),
            os.path.join(low_dir, '*.png')
        ]
        for pattern in low_patterns:
            low_images.extend(glob.glob(pattern))
        low_images = sorted(low_images)[:num_images]
    else:
        print(f"⚠️  Warning: Directory not found: {low_dir}")

    if os.path.exists(high_dir):
        high_patterns = [
            os.path.join(high_dir, '*.jpg'),
            os.path.join(high_dir, '*.jpeg'),
            os.path.join(high_dir, '*.png')
        ]
        for pattern in high_patterns:
            high_images.extend(glob.glob(pattern))
        high_images = sorted(high_images)[:num_images]
    else:
        print(f"⚠️  Warning: Directory not found: {high_dir}")

    print(f"\n✅ Loaded {len(low_images)} images from {low_dir}")
    print(f"✅ Loaded {len(high_images)} images from {high_dir}")

    if len(low_images) > 0:
        print(f"\nLow violence images:")
        for img in low_images:
            print(f"  - {os.path.basename(img)}")

    if len(high_images) > 0:
        print(f"\nHigh violence images:")
        for img in high_images:
            print(f"  - {os.path.basename(img)}")

    return low_images, high_images

# Load images
low_violence_paths, high_violence_paths = load_test_images(
    violence_low_dir,
    violence_high_dir,
    num_images=2
)

# Check if we have images
total_images = len(low_violence_paths) + len(high_violence_paths)
print(f"\nTotal images loaded: {total_images}")

if total_images == 0:
    print("\n" + "="*80)
    print("⚠️  ERROR: No test images found!")
    print("="*80)
    print("\nPlease ensure images are in:")
    print(f"  - {violence_low_dir}")
    print(f"  - {violence_high_dir}")
else:
    # ============================================================================
    # VISUALIZATION: Display Original Test Images
    # ============================================================================

    print("\n" + "=" * 80)
    print("Original Test Images")
    print("=" * 80)

    all_paths = low_violence_paths + high_violence_paths
    categories = ['Low Violence'] * len(low_violence_paths) + ['High Violence'] * len(high_violence_paths)

    num_images_to_show = min(4, len(all_paths))

    if num_images_to_show > 0:
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()

        for idx in range(num_images_to_show):
            ax = axes[idx]
            path = all_paths[idx]
            category = categories[idx]

            img = Image.open(path).convert('RGB')
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(f"{category}\n{os.path.basename(path)}",
                        fontsize=11, fontweight='bold')

        # Hide unused subplots
        for idx in range(num_images_to_show, 4):
            axes[idx].axis('off')

        plt.suptitle('Original Test Images', fontsize=14, fontweight='bold', y=0.98)
        plt.tight_layout()
        plt.savefig('plots/ablation_original_test_images.png', dpi=300, bbox_inches='tight')
        plt.show()

        print("✅ Original test images displayed")

    # ============================================================================
    # Train Model with KL + Classifier Loss Only
    # ============================================================================

    print("\n" + "=" * 80)
    print("Training Model with KL + Classifier Loss Only")
    print("=" * 80)

    # Create a fresh model
    ablation_model = PrototypeEncoder(
        input_dim=INPUT_DIM,
        latent_dim=LATENT_DIM,
        num_classes=NUM_CLASSES
    ).to(device)

    # Reset prototype EMA
    prototype_ema.reset()

    # Optimizer
    ablation_optimizer = optim.AdamW(
        ablation_model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-5  # Small weight decay for regularization
    )

    # Training parameters
    num_epochs_ablation = 20
    batch_losses = []

    # Training loop
    ablation_model.train()
    for epoch in range(num_epochs_ablation):
        epoch_loss = 0.0
        num_batches = 0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            ablation_optimizer.zero_grad()

            # Forward pass
            mu, logvar, z, distances, scores = ablation_model(batch_x)

            # Compute only KL and classification loss
            l_class = classification_loss(scores, batch_y, label_smoothing=0.1)
            l_kl = kl_divergence_loss(mu, logvar)

            # Total loss (NO SCL, NO EMA)
            total_loss = l_class + LAMBDA_KL * l_kl

            total_loss.backward()
            ablation_optimizer.step()

            epoch_loss += total_loss.item()
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        batch_losses.append(avg_loss)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs_ablation}], Loss: {avg_loss:.4f}")

    print("\n✅ Ablation model training complete")

    # ============================================================================
    # Evaluate on Test Set
    # ============================================================================

    print("\n" + "=" * 80)
    print("Evaluating Ablation Model")
    print("=" * 80)

    ablation_model.eval()

    all_labels = []
    all_predictions = []
    all_distances = []
    all_confidences = []
    all_z = []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            mu, logvar, z, distances, scores = ablation_model(batch_x)

            preds = torch.argmax(scores, dim=1)
            confidences = torch.max(scores, dim=1)[0]

            all_labels.extend(batch_y.cpu().numpy())
            all_predictions.extend(preds.cpu().numpy())
            all_distances.extend(distances.cpu().numpy())
            all_confidences.extend(confidences.cpu().numpy())
            all_z.append(z.cpu())

    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    all_distances = np.array(all_distances)
    all_confidences = np.array(all_confidences)
    all_z = torch.cat(all_z).numpy()

    # Calculate metrics
    accuracy = (all_predictions == all_labels).mean()
    print(f"\nTest Set Accuracy: {accuracy:.4f}")

    # ============================================================================
    # Process Test Images
    # ============================================================================

    print("\n" + "=" * 80)
    print("Processing Test Images")
    print("=" * 80)

    def process_test_image(image_path, model, clip_model, clip_processor):
        """Process a single test image and return predictions"""
        try:
            image = Image.open(image_path).convert('RGB')

            # Get CLIP embedding using clip_processor
            inputs = clip_processor(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                clip_emb = clip_model.get_image_features(**inputs).float()
                clip_emb = clip_emb / clip_emb.norm(dim=-1, keepdim=True)

            # Get model predictions
            with torch.no_grad():
                mu, logvar, z, distances, scores = model(clip_emb)
                pred_class = torch.argmax(scores, dim=1).item()
                confidence = torch.max(scores, dim=1)[0].item()
                distance_to_pred = distances[0, pred_class].item()

            return {
                'image': image,
                'pred_class': pred_class,
                'confidence': confidence,
                'distance': distance_to_pred,
                'all_distances': distances[0].cpu().numpy(),
                'all_scores': scores[0].cpu().numpy(),
                'z': z[0].cpu().numpy()
            }
        except Exception as e:
            print(f"Error processing {image_path}: {str(e)}")
            return None

    # Process all test images
    test_results_low = []
    test_results_high = []

    for path in low_violence_paths:
        result = process_test_image(path, ablation_model, clip_model, clip_processor)
        if result is not None:
            result['path'] = path
            result['category'] = 'Low Violence'
            test_results_low.append(result)
            print(f"\n✅ Low Violence Image: {os.path.basename(path)}")
            print(f"   Predicted: {CLASS_NAMES[result['pred_class']]}")
            print(f"   Confidence: {result['confidence']:.4f}")
            print(f"   Distance: {result['distance']:.4f}")
            print(f"   Scores: Normal={result['all_scores'][0]:.3f}, Violence={result['all_scores'][1]:.3f}, Sexual={result['all_scores'][2]:.3f}")

    for path in high_violence_paths:
        result = process_test_image(path, ablation_model, clip_model, clip_processor)
        if result is not None:
            result['path'] = path
            result['category'] = 'High Violence'
            test_results_high.append(result)
            print(f"\n✅ High Violence Image: {os.path.basename(path)}")
            print(f"   Predicted: {CLASS_NAMES[result['pred_class']]}")
            print(f"   Confidence: {result['confidence']:.4f}")
            print(f"   Distance: {result['distance']:.4f}")
            print(f"   Scores: Normal={result['all_scores'][0]:.3f}, Violence={result['all_scores'][1]:.3f}, Sexual={result['all_scores'][2]:.3f}")

    all_test_results = test_results_low + test_results_high

    print(f"\nTotal test results: {len(all_test_results)}")

    if len(all_test_results) > 0:
        # ========================================================================
        # VISUALIZATION: Training Loss Curve
        # ========================================================================

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(range(1, len(batch_losses) + 1), batch_losses, marker='o', linewidth=2, color='#3498db')
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('Loss', fontsize=12)
        ax.set_title('Ablation Study: Training Loss (KL + Classifier Only)',
                     fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('plots/ablation_kl_classifier_loss_curve.png', dpi=300, bbox_inches='tight')
        plt.show()

        # ========================================================================
        # VISUALIZATION: Test Images with Predictions
        # ========================================================================

        num_results = len(all_test_results)
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()

        for i in range(min(4, num_results)):
            result = all_test_results[i]
            ax = axes[i]
            ax.imshow(result['image'])
            ax.axis('off')

            title = f"{result['category']}\n"
            title += f"Pred: {CLASS_NAMES[result['pred_class']]}\n"
            title += f"Conf: {result['confidence']:.3f}, Dist: {result['distance']:.2f}"

            ax.set_title(title, fontsize=10, fontweight='bold')

        # Hide unused subplots
        for i in range(num_results, 4):
            axes[i].axis('off')

        plt.suptitle('Ablation Study: Test Image Predictions (KL + Classifier Only)',
                     fontsize=14, fontweight='bold', y=0.98)
        plt.tight_layout()
        plt.savefig('plots/ablation_kl_classifier_test_images.png', dpi=300, bbox_inches='tight')
        plt.show()

        # ========================================================================
        # VISUALIZATION: Distance Distribution per Class
        # ========================================================================

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        for class_idx in range(NUM_CLASSES):
            ax = axes[class_idx]
            class_mask = all_labels == class_idx
            class_distances = all_distances[class_mask][:, class_idx]

            ax.hist(class_distances, bins=30, alpha=0.7,
                    color=CLASS_COLORS[class_idx], edgecolor='black')
            ax.set_xlabel('Distance to Prototype', fontsize=10)
            ax.set_ylabel('Frequency', fontsize=10)
            ax.set_title(f'{CLASS_NAMES[class_idx]}', fontsize=12, fontweight='bold')
            ax.grid(True, alpha=0.3)

        plt.suptitle('Ablation Study: Distance Distribution (KL + Classifier Only)',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('plots/ablation_kl_classifier_distance_dist.png', dpi=300, bbox_inches='tight')
        plt.show()

        # ========================================================================
        # VISUALIZATION: Confidence vs Distance Scatter
        # ========================================================================

        fig, ax = plt.subplots(figsize=(10, 8))

        for class_idx in range(NUM_CLASSES):
            class_mask = all_labels == class_idx
            class_confidences = all_confidences[class_mask]
            class_distances = all_distances[class_mask][:, class_idx]

            ax.scatter(class_distances, class_confidences,
                      c=CLASS_COLORS[class_idx], label=CLASS_NAMES[class_idx],
                      alpha=0.5, s=50, edgecolors='black', linewidths=0.5)

        # Plot test images
        for result in all_test_results:
            pred_idx = result['pred_class']
            marker = 'o' if result['category'] == 'Low Violence' else '^'
            ax.scatter(result['distance'], result['confidence'],
                      c=CLASS_COLORS[pred_idx], marker=marker,
                      s=300, edgecolors='red', linewidths=3,
                      zorder=10)

        ax.set_xlabel('Distance to Prototype', fontsize=12)
        ax.set_ylabel('Confidence', fontsize=12)
        ax.set_title('Ablation Study: Confidence vs Distance (KL + Classifier Only)\n' +
                     'Red outline: Test images (circle=low, triangle=high)',
                     fontsize=14, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('plots/ablation_kl_classifier_conf_vs_dist.png', dpi=300, bbox_inches='tight')
        plt.show()

        # ========================================================================
        # VISUALIZATION: Score Comparison for Test Images
        # ========================================================================

        print("\n" + "=" * 80)
        print("Creating Score Comparison Chart...")
        print("=" * 80)

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()

        for i in range(min(4, num_results)):
            result = all_test_results[i]
            ax = axes[i]

            x = np.arange(NUM_CLASSES)
            scores = result['all_scores']

            print(f"\nPlotting result {i}:")
            print(f"  Scores: {scores}")
            print(f"  Pred class: {result['pred_class']}")

            # Create color list
            colors = [CLASS_COLORS[j] for j in range(NUM_CLASSES)]

            # Plot bars
            bars = ax.bar(x, scores, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

            # Highlight predicted class with red border
            bars[result['pred_class']].set_edgecolor('red')
            bars[result['pred_class']].set_linewidth(3)

            ax.set_xticks(x)
            ax.set_xticklabels(CLASS_NAMES, rotation=0, fontsize=9)
            ax.set_ylabel('Score', fontsize=10)
            ax.set_ylim([0, 1.0])  # Set y-axis limit
            ax.set_title(f"{result['category']} - {os.path.basename(result['path'])}\n" +
                        f"Pred: {CLASS_NAMES[result['pred_class']]} (Conf: {result['confidence']:.3f})",
                        fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')

        # Hide unused subplots
        for i in range(num_results, 4):
            axes[i].axis('off')

        plt.suptitle('Ablation Study: Classification Scores (KL + Classifier Only)',
                     fontsize=14, fontweight='bold', y=0.995)
        plt.tight_layout()
        plt.savefig('plots/ablation_kl_classifier_scores.png', dpi=300, bbox_inches='tight')
        plt.show()


        # ========================================================================
        # VISUALIZATION: Low vs High Violence Comparison
        # ========================================================================

        if len(test_results_low) > 0 and len(test_results_high) > 0:
            print("\n" + "=" * 80)
            print("Creating Low vs High Violence Comparison Charts...")
            print("=" * 80)

            # Use first image from each category for comparison
            low_result = test_results_low[0]
            high_result = test_results_high[0]

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

            # ----------------------------------------------------------------
            # Chart 1: Class Probability Comparison
            # ----------------------------------------------------------------

            x = np.arange(NUM_CLASSES)
            width = 0.35

            bars1 = ax1.bar(x - width/2, low_result['all_scores'], width,
                           label='Low Violence', color='blue', alpha=0.7, edgecolor='black')
            bars2 = ax1.bar(x + width/2, high_result['all_scores'], width,
                           label='High Violence', color='red', alpha=0.7, edgecolor='black')

            # Add value labels on bars
            for bar, val in zip(bars1, low_result['all_scores']):
                ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=10, color='blue', fontweight='bold')
            for bar, val in zip(bars2, high_result['all_scores']):
                ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=10, color='red', fontweight='bold')

            # Add arrows showing score change
            for idx in range(NUM_CLASSES):
                low_score = low_result['all_scores'][idx]
                high_score = high_result['all_scores'][idx]
                diff = high_score - low_score
                if abs(diff) > 0.01:
                    color = 'green' if diff > 0 else 'red'
                    symbol = '+' if diff > 0 else ''
                    ax1.annotate(f'{symbol}{diff:.3f}',
                                xy=(idx, max(low_score, high_score) + 0.08),
                                fontsize=11, color=color, ha='center', fontweight='bold')

            ax1.set_xlabel('Class', fontsize=12)
            ax1.set_ylabel('Probability Score', fontsize=12)
            ax1.set_title('Class Probability Comparison', fontsize=14, fontweight='bold')
            ax1.set_xticks(x)
            ax1.set_xticklabels(CLASS_NAMES, fontsize=11)
            ax1.set_ylim([0, 1.15])
            ax1.legend(fontsize=11, loc='upper right')
            ax1.grid(True, alpha=0.3, axis='y')

            # ----------------------------------------------------------------
            # Chart 2: Distance to Class Prototypes
            # ----------------------------------------------------------------

            bars1 = ax2.bar(x - width/2, low_result['all_distances'], width,
                           label='Low Violence', color='blue', alpha=0.7, edgecolor='black')
            bars2 = ax2.bar(x + width/2, high_result['all_distances'], width,
                           label='High Violence', color='red', alpha=0.7, edgecolor='black')

            # Add value labels on bars
            for bar, val in zip(bars1, low_result['all_distances']):
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                        f'{val:.2f}', ha='center', va='bottom', fontsize=10, color='blue', fontweight='bold')
            for bar, val in zip(bars2, high_result['all_distances']):
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                        f'{val:.2f}', ha='center', va='bottom', fontsize=10, color='red', fontweight='bold')

            ax2.set_xlabel('Class Prototype', fontsize=12)
            ax2.set_ylabel('Mahalanobis Distance', fontsize=12)
            ax2.set_title('Distance to Class Prototypes', fontsize=14, fontweight='bold')
            ax2.set_xticks(x)
            ax2.set_xticklabels(CLASS_NAMES, fontsize=11)
            ax2.legend(fontsize=11, loc='upper right')
            ax2.grid(True, alpha=0.3, axis='y')

            plt.tight_layout()
            plt.savefig('plots/ablation_low_vs_high_comparison.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\n✅ Low vs High violence comparison chart created")

            # ====================================================================
            # VISUALIZATION: t-SNE Latent Space with Test Images
            # ====================================================================

            print("\n" + "=" * 80)
            print("Creating t-SNE Latent Space Visualization...")
            print("=" * 80)

            # Collect latent vectors from test set
            print("Computing t-SNE projection...")

            # Combine test data latent vectors with our test images
            # Order: [all_z from test_loader, low_violence, high_violence]
            low_z = low_result['z'].reshape(1, -1)
            high_z = high_result['z'].reshape(1, -1)

            combined_z = np.vstack([all_z, low_z, high_z])
            print(f"  Test data: {all_z.shape[0]} points")
            print(f"  Query images: 2 points")
            print(f"  Combined: {combined_z.shape[0]} points")

            # Compute t-SNE
            from sklearn.manifold import TSNE
            tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(combined_z)-1), n_iter=1000)
            combined_tsne = tsne.fit_transform(combined_z)

            # Split back
            n_test = all_z.shape[0]
            test_tsne = combined_tsne[:n_test]
            low_tsne = combined_tsne[n_test]
            high_tsne = combined_tsne[n_test + 1]

            print("✅ t-SNE projection complete")

            # Get prototype positions (class centroids in t-SNE space)
            prototype_tsne = []
            for c in range(NUM_CLASSES):
                mask = all_labels == c
                if mask.sum() > 0:
                    centroid = test_tsne[mask].mean(axis=0)
                    prototype_tsne.append(centroid)
            prototype_tsne = np.array(prototype_tsne)

            # Create visualization
            fig, ax = plt.subplots(figsize=(12, 10))

            # Plot test data (background)
            for c in range(NUM_CLASSES):
                mask = all_labels == c
                ax.scatter(test_tsne[mask, 0], test_tsne[mask, 1],
                          c=CLASS_COLORS[c], alpha=0.3, s=30, label=CLASS_NAMES[c])

            # Plot prototypes (class centroids)
            for c in range(NUM_CLASSES):
                ax.scatter(prototype_tsne[c, 0], prototype_tsne[c, 1],
                          c=CLASS_COLORS[c], marker='*', s=500, edgecolors='black',
                          linewidths=2, zorder=10)

            # Plot low violence image
            ax.scatter(low_tsne[0], low_tsne[1], c='blue', marker='o', s=300,
                      edgecolors='black', linewidths=3, zorder=15, label='Low Violence')

            # Plot high violence image
            ax.scatter(high_tsne[0], high_tsne[1], c='red', marker='s', s=300,
                      edgecolors='black', linewidths=3, zorder=15, label='High Violence')

            # Draw arrow from low to high
            ax.annotate('', xy=(high_tsne[0], high_tsne[1]),
                       xytext=(low_tsne[0], low_tsne[1]),
                       arrowprops=dict(arrowstyle='->', color='purple', lw=3,
                                      connectionstyle='arc3,rad=0.1'))

            # Calculate drift magnitude
            latent_drift = np.linalg.norm(high_z - low_z)
            tsne_drift = np.linalg.norm(high_tsne - low_tsne)

            # Add drift label
            mid_point = (low_tsne + high_tsne) / 2
            ax.annotate(f'Drift: {latent_drift:.2f}', xy=mid_point, fontsize=12,
                       color='purple', fontweight='bold',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

            ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
            ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
            ax.set_title('Ablation Study: t-SNE Latent Space Visualization\n' +
                        'Low Violence (blue circle) vs High Violence (red square)',
                        fontsize=14, fontweight='bold')
            ax.legend(loc='best', fontsize=10)
            ax.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig('plots/ablation_tsne_latent_space.png', dpi=300, bbox_inches='tight')
            plt.show()

            print("\n✅ t-SNE latent space visualization created")

            # Print drift summary
            print("\n" + "=" * 80)
            print("LATENT SPACE DRIFT SUMMARY")
            print("=" * 80)
            print(f"Latent Space Drift:   {latent_drift:.4f}")
            print(f"t-SNE Space Drift:    {tsne_drift:.4f}")
            print(f"\nLow Violence Image:")
            print(f"  File: {os.path.basename(low_result['path'])}")
            print(f"  Predicted: {CLASS_NAMES[low_result['pred_class']]} (Conf: {low_result['confidence']:.3f})")
            print(f"\nHigh Violence Image:")
            print(f"  File: {os.path.basename(high_result['path'])}")
            print(f"  Predicted: {CLASS_NAMES[high_result['pred_class']]} (Conf: {high_result['confidence']:.3f})")
            print("=" * 80)


            # Print summary
            print("\n" + "=" * 80)
            print("COMPARISON SUMMARY")
            print("=" * 80)
            print(f"Low Violence Image:  {os.path.basename(low_result['path'])}")
            print(f"  Predicted: {CLASS_NAMES[low_result['pred_class']]} (Conf: {low_result['confidence']:.3f})")
            print(f"  Scores: Normal={low_result['all_scores'][0]:.3f}, Violence={low_result['all_scores'][1]:.3f}, Sexual={low_result['all_scores'][2]:.3f}")
            print(f"\nHigh Violence Image: {os.path.basename(high_result['path'])}")
            print(f"  Predicted: {CLASS_NAMES[high_result['pred_class']]} (Conf: {high_result['confidence']:.3f})")
            print(f"  Scores: Normal={high_result['all_scores'][0]:.3f}, Violence={high_result['all_scores'][1]:.3f}, Sexual={high_result['all_scores'][2]:.3f}")
            print("=" * 80)

        print("\n✅ Score comparison chart created")

        # ====================================================================
        # Summary
        # ====================================================================

        print("\n" + "=" * 80)
        print("✅ Ablation Study Complete")
        print("=" * 80)
        print(f"\nModel Accuracy: {accuracy:.4f}")
        print(f"Test Images Processed: {len(all_test_results)}")
        print("\nPlots saved:")
        print("  • plots/ablation_original_test_images.png")
        print("  • plots/ablation_kl_classifier_loss_curve.png")
        print("  • plots/ablation_kl_classifier_test_images.png")
        print("  • plots/ablation_kl_classifier_distance_dist.png")
        print("  • plots/ablation_kl_classifier_conf_vs_dist.png")
        print("  • plots/ablation_kl_classifier_scores.png")
    else:
        print("\n⚠️  No test results to visualize")

In [ ]:
# ============================================================================
# SEMANTIC DRIFT VISUALIZATION: Compare Two Images
# ============================================================================
#
# This cell visualizes the semantic drift between an original image and a
# manipulated/transformed image in the learned latent space.
#
# NOTE: Uses the already trained model (trained_model) and test_loader from
#       previous cells. No re-training required.
#
# Features:
# 1. Encode both images using CLIP + trained model
# 2. Plot positions on t-SNE (all points transformed together for accuracy)
# 3. Show probability distributions for each class
# 4. Draw arrow indicating drift direction and magnitude
# ============================================================================

import matplotlib.patches as mpatches
from PIL import Image

# ============================================================================
# Helper Functions
# ============================================================================

def load_and_encode_image(image_path, clip_model, clip_processor, model, device):
    """
    Load an image, extract CLIP embedding, and encode with trained model.
    """
    # Load image
    image = Image.open(image_path).convert('RGB')

    # Extract CLIP embedding
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        clip_embedding = clip_model.get_image_features(**inputs)
        clip_embedding = clip_embedding / clip_embedding.norm(dim=-1, keepdim=True)
        clip_embedding = clip_embedding.float()
        clip_embedding = clip_embedding.squeeze(0)

    # Encode with trained model
    model.eval()
    with torch.no_grad():
        x = clip_embedding.unsqueeze(0)
        mu, logvar, z, distances, scores = model(x)

    return {
        'image': image,
        'clip_embedding': clip_embedding.cpu().numpy(),
        'mu': mu.squeeze(0).cpu().numpy(),
        'z': z.squeeze(0).cpu().numpy(),
        'distances': distances.squeeze(0).cpu().numpy(),
        'scores': scores.squeeze(0).cpu().numpy()
    }


def visualize_semantic_drift(original_path, manipulated_path,
                              model, clip_model, clip_processor,
                              data_loader, device,
                              save_path='plots/semantic_drift_comparison.png'):
    """
    Visualize semantic drift between original and manipulated images.

    All points (training data + query images) are transformed together
    in a single t-SNE for accurate positioning.

    Args:
        original_path: Path to original image
        manipulated_path: Path to manipulated/transformed image
        model: Trained PrototypeEncoder model
        clip_model: CLIP model for embedding extraction
        clip_processor: CLIP processor
        data_loader: DataLoader with test data
        device: torch device
        save_path: Path to save the visualization
    """

    print("="*70)
    print("SEMANTIC DRIFT VISUALIZATION")
    print("="*70)
    print(f"Original image: {original_path}")
    print(f"Manipulated image: {manipulated_path}")
    print()

    # 1. Encode query images first
    print("Encoding query images...")
    original_data = load_and_encode_image(original_path, clip_model, clip_processor, model, device)
    manipulated_data = load_and_encode_image(manipulated_path, clip_model, clip_processor, model, device)

    # 2. Collect all latent vectors from test data
    print("Computing latent vectors from test data...")
    model.eval()
    all_z = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in tqdm(data_loader, desc="Encoding"):
            batch_x = batch_x.to(device)
            mu, logvar, z, distances, scores = model(batch_x)
            all_z.append(z.cpu().numpy())
            all_labels.append(batch_y.numpy())

    all_z = np.concatenate(all_z, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # 3. Combine ALL points for single t-SNE transformation
    # Order: [test_data, original_query, manipulated_query]
    print("Combining all points for t-SNE...")
    print(f"  Test data: {all_z.shape[0]} points")
    print(f"  Query images: 2 points")

    combined = np.vstack([
        all_z,                              # Test data
        original_data['z'].reshape(1, -1),  # Original query
        manipulated_data['z'].reshape(1, -1) # Manipulated query
    ])
    print(f"  Combined: {combined.shape[0]} points")

    # 4. Single t-SNE transformation for ALL points
    print("Computing t-SNE projection (all points together)...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(combined)-1), n_iter=1000)
    combined_tsne = tsne.fit_transform(combined)

    # 5. Split back
    n_test = all_z.shape[0]
    test_tsne = combined_tsne[:n_test]
    original_2d = combined_tsne[n_test]
    manipulated_2d = combined_tsne[n_test + 1]

    print("t-SNE complete!")

    # Get prototype positions (class centroids in t-SNE space)
    prototype_2d = []
    for c in range(NUM_CLASSES):
        mask = all_labels == c
        if mask.sum() > 0:
            centroid = test_tsne[mask].mean(axis=0)
            prototype_2d.append(centroid)
    prototype_2d = np.array(prototype_2d)

    # 6. Create visualization
    fig = plt.figure(figsize=(20, 8))

    # =========================================================================
    # Plot 1: Latent Space with Drift Arrow
    # =========================================================================
    ax1 = fig.add_subplot(1, 3, 1)

    # Plot training data (background)
    for c in range(NUM_CLASSES):
        mask = all_labels == c
        ax1.scatter(test_tsne[mask, 0], test_tsne[mask, 1],
                   c=CLASS_COLORS[c], alpha=0.3, s=30, label=CLASS_NAMES[c])

    # Plot prototypes (class centroids)
    for c in range(NUM_CLASSES):
        ax1.scatter(prototype_2d[c, 0], prototype_2d[c, 1],
                   c=CLASS_COLORS[c], marker='*', s=400, edgecolors='black',
                   linewidths=2, zorder=10)

    # Plot original image position
    ax1.scatter(original_2d[0], original_2d[1], c='blue', marker='o', s=200,
               edgecolors='black', linewidths=2, zorder=15, label='Original')

    # Plot manipulated image position
    ax1.scatter(manipulated_2d[0], manipulated_2d[1], c='red', marker='s', s=200,
               edgecolors='black', linewidths=2, zorder=15, label='Manipulated')

    # Draw drift arrow
    ax1.annotate('', xy=(manipulated_2d[0], manipulated_2d[1]),
                xytext=(original_2d[0], original_2d[1]),
                arrowprops=dict(arrowstyle='->', color='purple', lw=3,
                               connectionstyle='arc3,rad=0.1'))

    # Calculate drift magnitude (in latent space)
    latent_drift = np.linalg.norm(manipulated_data['z'] - original_data['z'])
    tsne_drift = np.linalg.norm(manipulated_2d - original_2d)

    # Add drift label
    mid_point = (original_2d + manipulated_2d) / 2
    ax1.annotate(f'Drift: {latent_drift:.2f}', xy=mid_point, fontsize=12,
                color='purple', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax1.set_xlabel('t-SNE Dimension 1', fontsize=12)
    ax1.set_ylabel('t-SNE Dimension 2', fontsize=12)
    ax1.set_title('Latent Space: Semantic Drift Visualization', fontsize=14, fontweight='bold')
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3)

    # =========================================================================
    # Plot 2: Class Probability Comparison (Bar Chart)
    # =========================================================================
    ax2 = fig.add_subplot(1, 3, 2)

    x = np.arange(NUM_CLASSES)
    width = 0.35

    bars1 = ax2.bar(x - width/2, original_data['scores'], width,
                    label='Original', color='blue', alpha=0.7)
    bars2 = ax2.bar(x + width/2, manipulated_data['scores'], width,
                    label='Manipulated', color='red', alpha=0.7)

    # Add value labels
    for bar, val in zip(bars1, original_data['scores']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, color='blue')
    for bar, val in zip(bars2, manipulated_data['scores']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, color='red')

    # Add arrows showing score change
    for i in range(NUM_CLASSES):
        orig_score = original_data['scores'][i]
        manip_score = manipulated_data['scores'][i]
        diff = manip_score - orig_score
        if abs(diff) > 0.01:
            color = 'green' if diff > 0 else 'red'
            symbol = '+' if diff > 0 else '-'
            ax2.annotate(f'{symbol}{abs(diff):.3f}',
                        xy=(i, max(orig_score, manip_score) + 0.08),
                        fontsize=11, color=color, ha='center', fontweight='bold')

    ax2.set_xlabel('Class', fontsize=12)
    ax2.set_ylabel('Probability Score', fontsize=12)
    ax2.set_title('Class Probability Comparison', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(CLASS_NAMES, fontsize=11)
    ax2.set_ylim([0, 1.15])
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')

    # =========================================================================
    # Plot 3: Distance to Prototypes Comparison
    # =========================================================================
    ax3 = fig.add_subplot(1, 3, 3)

    bars1 = ax3.bar(x - width/2, original_data['distances'], width,
                    label='Original', color='blue', alpha=0.7)
    bars2 = ax3.bar(x + width/2, manipulated_data['distances'], width,
                    label='Manipulated', color='red', alpha=0.7)

    # Add value labels
    for bar, val in zip(bars1, original_data['distances']):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10, color='blue')
    for bar, val in zip(bars2, manipulated_data['distances']):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10, color='red')

    ax3.set_xlabel('Class Prototype', fontsize=12)
    ax3.set_ylabel('Mahalanobis Distance', fontsize=12)
    ax3.set_title('Distance to Class Prototypes', fontsize=14, fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels(CLASS_NAMES, fontsize=11)
    ax3.legend(fontsize=11)
    ax3.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()

    # =========================================================================
    # Summary
    # =========================================================================
    original_pred = CLASS_NAMES[np.argmax(original_data['scores'])]
    manipulated_pred = CLASS_NAMES[np.argmax(manipulated_data['scores'])]
    original_conf = np.max(original_data['scores'])
    manipulated_conf = np.max(manipulated_data['scores'])

    score_diff = manipulated_data['scores'] - original_data['scores']
    max_increase_class = CLASS_NAMES[np.argmax(score_diff)]
    max_decrease_class = CLASS_NAMES[np.argmin(score_diff)]

    summary_text = f"""
====================================================================
                    SEMANTIC DRIFT SUMMARY
====================================================================
  Original:     {original_pred:<12} (conf: {original_conf:.3f})
  Manipulated:  {manipulated_pred:<12} (conf: {manipulated_conf:.3f})
--------------------------------------------------------------------
  Drift Direction:
    + Increased: {max_increase_class:<12} ({score_diff.max():+.3f})
    - Decreased: {max_decrease_class:<12} ({score_diff.min():+.3f})
--------------------------------------------------------------------
  Latent Space Drift: {latent_drift:.4f}
  t-SNE Space Drift:  {tsne_drift:.4f}
  Prediction Changed: {'Yes' if original_pred != manipulated_pred else 'No'}
====================================================================
"""

    print(summary_text)

    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"Visualization saved to: {save_path}")

    return {
        'original': original_data,
        'manipulated': manipulated_data,
        'latent_drift': latent_drift,
        'tsne_drift': tsne_drift,
        'original_pred': original_pred,
        'manipulated_pred': manipulated_pred
    }


# ============================================================================
# USAGE
# ============================================================================

# Example paths (UPDATE THESE WITH YOUR ACTUAL IMAGE PATHS)
ORIGINAL_IMAGE_PATH = '/content/drive/MyDrive/vdsh_watermarking/inference_test/violence_low/original1.jpg'
MANIPULATED_IMAGE_PATH = '/content/drive/MyDrive/vdsh_watermarking/inference_test/violence_high/manipulated1.png'

# Visualize semantic drift
# Uncomment below to run the visualization:
result = visualize_semantic_drift(
    original_path=ORIGINAL_IMAGE_PATH,
    manipulated_path=MANIPULATED_IMAGE_PATH,
    model=trained_model,
    clip_model=clip_model,
    clip_processor=clip_processor,
    data_loader=test_loader,
    device=device
)

print("="*70)
print("SEMANTIC DRIFT VISUALIZATION READY")
print("="*70)
print()
print("Method: All points (test data + query images) are transformed")
print("        together in a single t-SNE for accurate positioning.")
print()
print("To visualize drift between two images:")
print("  1. Update ORIGINAL_IMAGE_PATH and MANIPULATED_IMAGE_PATH")
print("  2. Uncomment the visualization call above")
print("  3. Run this cell")
print()
print("Note: Each run recomputes t-SNE (~30 sec) for accuracy.")
print("="*70)


In [ ]:
# ============================================================================
# SEMANTIC DRIFT VISUALIZATION: Compare Three Images (Sequential Drift)
# ============================================================================
#
# This cell visualizes sequential semantic drift across THREE images:
# Image 1 → Image 2 → Image 3
#
# Use case: Track how an image changes through multiple manipulation stages
# e.g., Original → Manipulated Low → Manipulated High
#
# Features:
# 1. Encode all three images using CLIP + trained model
# 2. Plot positions on t-SNE (all points transformed together)
# 3. Show probability distributions for each class
# 4. Draw TWO arrows showing sequential drift (1→2→3)
# ============================================================================

import matplotlib.patches as mpatches
from PIL import Image

def load_and_encode_image(image_path, clip_model, clip_processor, model, device):
    """
    Load an image, extract CLIP embedding, and encode with trained model.
    """
    image = Image.open(image_path).convert('RGB')

    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        clip_embedding = clip_model.get_image_features(**inputs)
        clip_embedding = clip_embedding / clip_embedding.norm(dim=-1, keepdim=True)
        clip_embedding = clip_embedding.float()
        clip_embedding = clip_embedding.squeeze(0)

    model.eval()
    with torch.no_grad():
        x = clip_embedding.unsqueeze(0)
        mu, logvar, z, distances, scores = model(x)

    return {
        'image': image,
        'clip_embedding': clip_embedding.cpu().numpy(),
        'mu': mu.squeeze(0).cpu().numpy(),
        'z': z.squeeze(0).cpu().numpy(),
        'distances': distances.squeeze(0).cpu().numpy(),
        'scores': scores.squeeze(0).cpu().numpy()
    }


def visualize_sequential_drift(image1_path, image2_path, image3_path,
                                model, clip_model, clip_processor,
                                data_loader, device,
                                labels=None,
                                save_path='plots/sequential_drift_comparison.png'):
    """
    Visualize sequential semantic drift across three images: 1 → 2 → 3

    Args:
        image1_path: Path to first image (e.g., original)
        image2_path: Path to second image (e.g., Manipulated Low)
        image3_path: Path to third image (e.g., Manipulated High)
        model: Trained PrototypeEncoder model
        clip_model: CLIP model for embedding extraction
        clip_processor: CLIP processor
        data_loader: DataLoader with test data
        device: torch device
        labels: Optional list of 3 labels for the images (default: ['Image 1', 'Image 2', 'Image 3'])
        save_path: Path to save the visualization
    """

    if labels is None:
        labels = ['Image 1', 'Image 2', 'Image 3']

    print("="*70)
    print("SEQUENTIAL DRIFT VISUALIZATION (3 Images)")
    print("="*70)
    print(f"{labels[0]}: {image1_path}")
    print(f"{labels[1]}: {image2_path}")
    print(f"{labels[2]}: {image3_path}")
    print()

    # 1. Encode all three query images
    print("Encoding query images...")
    data1 = load_and_encode_image(image1_path, clip_model, clip_processor, model, device)
    data2 = load_and_encode_image(image2_path, clip_model, clip_processor, model, device)
    data3 = load_and_encode_image(image3_path, clip_model, clip_processor, model, device)

    # 2. Collect all latent vectors from test data
    print("Computing latent vectors from test data...")
    model.eval()
    all_z = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in tqdm(data_loader, desc="Encoding"):
            batch_x = batch_x.to(device)
            mu, logvar, z, distances, scores = model(batch_x)
            all_z.append(z.cpu().numpy())
            all_labels.append(batch_y.numpy())

    all_z = np.concatenate(all_z, axis=0)
    all_labels_arr = np.concatenate(all_labels, axis=0)

    # 3. Combine ALL points for single t-SNE transformation
    print("Combining all points for t-SNE...")
    print(f"  Test data: {all_z.shape[0]} points")
    print(f"  Query images: 3 points")

    combined = np.vstack([
        all_z,
        data1['z'].reshape(1, -1),
        data2['z'].reshape(1, -1),
        data3['z'].reshape(1, -1)
    ])
    print(f"  Combined: {combined.shape[0]} points")

    # 4. Single t-SNE transformation
    print("Computing t-SNE projection (all points together)...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(combined)-1), n_iter=1000)
    combined_tsne = tsne.fit_transform(combined)

    # 5. Split back
    n_test = all_z.shape[0]
    test_tsne = combined_tsne[:n_test]
    pos1_2d = combined_tsne[n_test]
    pos2_2d = combined_tsne[n_test + 1]
    pos3_2d = combined_tsne[n_test + 2]

    print("t-SNE complete!")

    # Get prototype positions
    prototype_2d = []
    for c in range(NUM_CLASSES):
        mask = all_labels_arr == c
        if mask.sum() > 0:
            centroid = test_tsne[mask].mean(axis=0)
            prototype_2d.append(centroid)
    prototype_2d = np.array(prototype_2d)

    # Calculate drift magnitudes
    drift_1_to_2 = np.linalg.norm(data2['z'] - data1['z'])
    drift_2_to_3 = np.linalg.norm(data3['z'] - data2['z'])
    drift_1_to_3 = np.linalg.norm(data3['z'] - data1['z'])

    # 6. Create visualization
    fig = plt.figure(figsize=(20, 8))

    # =========================================================================
    # Plot 1: Latent Space with Sequential Drift Arrows
    # =========================================================================
    ax1 = fig.add_subplot(1, 3, 1)

    # Plot training data (background)
    for c in range(NUM_CLASSES):
        mask = all_labels_arr == c
        ax1.scatter(test_tsne[mask, 0], test_tsne[mask, 1],
                   c=CLASS_COLORS[c], alpha=0.3, s=30, label=CLASS_NAMES[c])

    # Plot prototypes
    for c in range(NUM_CLASSES):
        ax1.scatter(prototype_2d[c, 0], prototype_2d[c, 1],
                   c=CLASS_COLORS[c], marker='*', s=400, edgecolors='black',
                   linewidths=2, zorder=10)

    # Plot three image positions with different markers
    ax1.scatter(pos1_2d[0], pos1_2d[1], c='blue', marker='o', s=250,
               edgecolors='black', linewidths=2, zorder=15, label=labels[0])
    ax1.scatter(pos2_2d[0], pos2_2d[1], c='orange', marker='s', s=250,
               edgecolors='black', linewidths=2, zorder=15, label=labels[1])
    ax1.scatter(pos3_2d[0], pos3_2d[1], c='red', marker='^', s=250,
               edgecolors='black', linewidths=2, zorder=15, label=labels[2])

    # Draw drift arrow 1→2 (blue to orange)
    ax1.annotate('', xy=(pos2_2d[0], pos2_2d[1]),
                xytext=(pos1_2d[0], pos1_2d[1]),
                arrowprops=dict(arrowstyle='->', color='darkblue', lw=3,
                               connectionstyle='arc3,rad=0.1'))

    # Draw drift arrow 2→3 (orange to red)
    ax1.annotate('', xy=(pos3_2d[0], pos3_2d[1]),
                xytext=(pos2_2d[0], pos2_2d[1]),
                arrowprops=dict(arrowstyle='->', color='darkred', lw=3,
                               connectionstyle='arc3,rad=0.1'))

    # Add drift labels
    mid_1_2 = (pos1_2d + pos2_2d) / 2
    mid_2_3 = (pos2_2d + pos3_2d) / 2

    ax1.annotate(f'{drift_1_to_2:.2f}', xy=mid_1_2, fontsize=11,
                color='darkblue', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax1.annotate(f'{drift_2_to_3:.2f}', xy=mid_2_3, fontsize=11,
                color='darkred', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax1.set_xlabel('t-SNE Dimension 1', fontsize=12)
    ax1.set_ylabel('t-SNE Dimension 2', fontsize=12)
    ax1.set_title('Latent Space: Sequential Drift', fontsize=14, fontweight='bold')
    ax1.legend(loc='best', fontsize=9)
    ax1.grid(True, alpha=0.3)

    # =========================================================================
    # Plot 2: Class Probability Comparison (3 bars)
    # =========================================================================
    ax2 = fig.add_subplot(1, 3, 2)

    x = np.arange(NUM_CLASSES)
    width = 0.25

    bars1 = ax2.bar(x - width, data1['scores'], width,
                    label=labels[0], color='blue', alpha=0.7)
    bars2 = ax2.bar(x, data2['scores'], width,
                    label=labels[1], color='orange', alpha=0.7)
    bars3 = ax2.bar(x + width, data3['scores'], width,
                    label=labels[2], color='red', alpha=0.7)

    # Add value labels
    for bar, val in zip(bars1, data1['scores']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8, color='blue')
    for bar, val in zip(bars2, data2['scores']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8, color='orange')
    for bar, val in zip(bars3, data3['scores']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8, color='red')

    ax2.set_xlabel('Class', fontsize=12)
    ax2.set_ylabel('Probability Score', fontsize=12)
    ax2.set_title('Class Probability Comparison', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(CLASS_NAMES, fontsize=11)
    ax2.set_ylim([0, 1.15])
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3, axis='y')

    # =========================================================================
    # Plot 3: Distance to Prototypes Comparison
    # =========================================================================
    ax3 = fig.add_subplot(1, 3, 3)

    bars1 = ax3.bar(x - width, data1['distances'], width,
                    label=labels[0], color='blue', alpha=0.7)
    bars2 = ax3.bar(x, data2['distances'], width,
                    label=labels[1], color='orange', alpha=0.7)
    bars3 = ax3.bar(x + width, data3['distances'], width,
                    label=labels[2], color='red', alpha=0.7)

    # Add value labels
    for bar, val in zip(bars1, data1['distances']):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8, color='blue')
    for bar, val in zip(bars2, data2['distances']):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8, color='orange')
    for bar, val in zip(bars3, data3['distances']):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8, color='red')

    ax3.set_xlabel('Class Prototype', fontsize=12)
    ax3.set_ylabel('Mahalanobis Distance', fontsize=12)
    ax3.set_title('Distance to Class Prototypes', fontsize=14, fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels(CLASS_NAMES, fontsize=11)
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()

    # =========================================================================
    # Summary
    # =========================================================================
    pred1 = CLASS_NAMES[np.argmax(data1['scores'])]
    pred2 = CLASS_NAMES[np.argmax(data2['scores'])]
    pred3 = CLASS_NAMES[np.argmax(data3['scores'])]
    conf1 = np.max(data1['scores'])
    conf2 = np.max(data2['scores'])
    conf3 = np.max(data3['scores'])

    summary_text = f"""
====================================================================
              SEQUENTIAL DRIFT SUMMARY (3 Images)
====================================================================
  {labels[0]:<15} → {pred1:<12} (conf: {conf1:.3f})
  {labels[1]:<15} → {pred2:<12} (conf: {conf2:.3f})
  {labels[2]:<15} → {pred3:<12} (conf: {conf3:.3f})
--------------------------------------------------------------------
  Drift Magnitudes (Latent Space):
    {labels[0]} → {labels[1]}:  {drift_1_to_2:.4f}
    {labels[1]} → {labels[2]}:  {drift_2_to_3:.4f}
    {labels[0]} → {labels[2]}:  {drift_1_to_3:.4f} (total)
--------------------------------------------------------------------
  Class Changes: {pred1} → {pred2} → {pred3}
====================================================================
"""

    print(summary_text)

    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"Visualization saved to: {save_path}")

    return {
        'data1': data1,
        'data2': data2,
        'data3': data3,
        'drift_1_to_2': drift_1_to_2,
        'drift_2_to_3': drift_2_to_3,
        'drift_1_to_3': drift_1_to_3,
        'predictions': [pred1, pred2, pred3]
    }


# ============================================================================
# USAGE
# ============================================================================

# Example paths (UPDATE THESE WITH YOUR ACTUAL IMAGE PATHS)
IMAGE1_PATH = '/content/drive/MyDrive/vdsh_watermarking/inference_test/original3.jpeg'
IMAGE2_PATH =  '/content/drive/MyDrive/vdsh_watermarking/inference_test/watermarked3.png'
IMAGE3_PATH =  '/content/drive/MyDrive/vdsh_watermarking/inference_test/manipulated3.png'

# Custom labels for the three images
# IMAGE_LABELS = ['Original', 'Manipulated Low', 'Manipulated High']
IMAGE_LABELS = ['Original', 'Watermarked', 'Manipulated']
# Visualize sequential drift
# Uncomment below to run the visualization:
result = visualize_sequential_drift(
    image1_path=IMAGE1_PATH,
    image2_path=IMAGE2_PATH,
    image3_path=IMAGE3_PATH,
    model=trained_model,
    clip_model=clip_model,
    clip_processor=clip_processor,
    data_loader=test_loader,
    device=device,
    labels=IMAGE_LABELS
)

print("="*70)
print("SEQUENTIAL DRIFT VISUALIZATION READY (3 Images)")
print("="*70)
print()
print("Visualizes: Image 1 → Image 2 → Image 3")
print("            (e.g., Original → Manipulated Low → Manipulated High)")
print()
print("To use:")
print("  1. Update IMAGE1_PATH, IMAGE2_PATH, IMAGE3_PATH")
print("  2. Update IMAGE_LABELS (optional)")
print("  3. Uncomment the visualization call above")
print("  4. Run this cell")
print("="*70)

